In [4]:
# Step 1: Import required libraries

import ee
import geemap
import folium
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

print("All libraries imported successfully!")


All libraries imported successfully!


In [5]:
# Step 2: Connect Google Earth Engine

PROJECT_ID = "punjab-geoguardian-515011"

ee.Initialize(project=PROJECT_ID)

print("Google Earth Engine Connected Successfully!")

Google Earth Engine Connected Successfully!


In [6]:
# Step 3: Define Area of Interest (AOI)

aoi = ee.Geometry.Rectangle([
    74.20, 31.30,
    74.28, 31.38
])

area_km2 = aoi.area().divide(1_000_000).getInfo()

print("AOI: Lahore Peri-Urban Expansion Zone")
print("Location: Raiwind Road, Lahore")
print("Approximate AOI Area:", round(area_km2, 2), "km²")

AOI: Lahore Peri-Urban Expansion Zone
Location: Raiwind Road, Lahore
Approximate AOI Area: 67.59 km²


In [7]:
# Step 4: Visualize Area of Interest

Map = geemap.Map(center=[31.34, 74.24], zoom=11)

Map.addLayer(
    aoi,
    {"color": "red"},
    "Lahore Raiwind Road AOI"
)

Map.centerObject(aoi, 11)

Map

Map(center=[31.340000534082467, 74.23999999999776], controls=(WidgetControl(options=['position', 'transparent_…

In [8]:
# Step 4: Display AOI on a basemap

Map = geemap.Map(
    center=[31.34, 74.24],
    zoom=11
)

Map.add_basemap("Google Maps")

Map.addLayer(
    aoi,
    {"color": "red"},
    "Lahore Raiwind Road AOI"
)

Map

Basemap can only be one of the following:
  OpenStreetMap
  Esri.WorldStreetMap
  Esri.WorldImagery
  Esri.WorldTopoMap
  FWS NWI Wetlands
  FWS NWI Wetlands Raster
  NLCD 2021 CONUS Land Cover
  NLCD 2019 CONUS Land Cover
  NLCD 2016 CONUS Land Cover
  NLCD 2013 CONUS Land Cover
  NLCD 2011 CONUS Land Cover
  NLCD 2008 CONUS Land Cover
  NLCD 2006 CONUS Land Cover
  NLCD 2004 CONUS Land Cover
  NLCD 2001 CONUS Land Cover
  USGS NAIP Imagery
  USGS NAIP Imagery False Color
  USGS NAIP Imagery NDVI
  USGS Hydrography
  USGS 3DEP Elevation
  ESA Worldcover 2020
  ESA Worldcover 2020 S2 FCC
  ESA Worldcover 2020 S2 TCC
  ESA Worldcover 2021
  ESA Worldcover 2021 S2 FCC
  ESA Worldcover 2021 S2 TCC
  BaseMapDE.Color
  BaseMapDE.Grey
  BasemapAT.basemap
  BasemapAT.grau
  BasemapAT.highdpi
  BasemapAT.orthofoto
  BasemapAT.overlay
  BasemapAT.surface
  BasemapAT.terrain
  CyclOSM
  Esri.AntarcticBasemap
  Esri.AntarcticImagery
  Esri.ArcticImagery
  Esri.ArcticOceanBase
  Esri.ArcticOceanRe

Map(center=[31.34, 74.24], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [9]:
# Step 4: Test map display using Folium

import folium
from IPython.display import display

test_map = folium.Map(
    location=[31.34, 74.24],
    zoom_start=11,
    tiles="OpenStreetMap"
)

folium.Rectangle(
    bounds=[
        [31.30, 74.20],
        [31.38, 74.28]
    ],
    color="red",
    fill=False
).add_to(test_map)

display(test_map)

In [10]:
# Step 5: Collect Sentinel-2 Satellite Images

# Sentinel-2 Surface Reflectance Harmonized Dataset
sentinel_collection = "COPERNICUS/S2_SR_HARMONIZED"

# 2019 satellite images
sentinel_2019 = (
    ee.ImageCollection(sentinel_collection)
    .filterBounds(aoi)
    .filterDate("2019-10-01", "2020-01-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

# 2025 satellite images
sentinel_2025 = (
    ee.ImageCollection(sentinel_collection)
    .filterBounds(aoi)
    .filterDate("2025-10-01", "2026-01-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

# Count available images
count_2019 = sentinel_2019.size().getInfo()
count_2025 = sentinel_2025.size().getInfo()

print("Sentinel-2 Data Collection")
print("--------------------------")
print("2019 Images:", count_2019)
print("2025 Images:", count_2025)
print("Cloud Cover Filter: Less than 20%")
print("Season: October to December")

Sentinel-2 Data Collection
--------------------------
2019 Images: 14
2025 Images: 32
Cloud Cover Filter: Less than 20%
Season: October to December


In [11]:
# Step 6: Cloud Masking and Median Composite

def mask_clouds(image):

    scl = image.select("SCL")

    mask = (
        scl.neq(3)      # Cloud shadows
        .And(scl.neq(8))    # Medium probability clouds
        .And(scl.neq(9))    # High probability clouds
        .And(scl.neq(10))   # Cirrus clouds
        .And(scl.neq(11))   # Snow and ice
    )

    return image.updateMask(mask)


# Apply cloud masking to 2019 images
sentinel_2019_clean = sentinel_2019.map(mask_clouds)

# Apply cloud masking to 2025 images
sentinel_2025_clean = sentinel_2025.map(mask_clouds)


# Create median composites
image_2019 = sentinel_2019_clean.median().clip(aoi)

image_2025 = sentinel_2025_clean.median().clip(aoi)


print("Cloud masking completed successfully!")
print("2019 median composite created.")
print("2025 median composite created.")

Cloud masking completed successfully!
2019 median composite created.
2025 median composite created.


In [12]:
# Step 7: Create RGB Satellite Maps

rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 3000
}

# Create Folium map
rgb_map = geemap.Map(center=[31.34, 74.24], zoom=11)

# Add 2019 satellite image
rgb_map.addLayer(
    image_2019,
    rgb_vis,
    "RGB Satellite Image 2019"
)

# Add 2025 satellite image
rgb_map.addLayer(
    image_2025,
    rgb_vis,
    "RGB Satellite Image 2025"
)

# Add AOI boundary
rgb_map.addLayer(
    aoi,
    {"color": "red"},
    "AOI Boundary"
)

rgb_map.addLayerControl()

rgb_map

Map(center=[31.34, 74.24], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [13]:
# Step 8: Calculate NDVI for 2019 and 2025

ndvi_2019 = (
    image_2019
    .normalizedDifference(["B8", "B4"])
    .rename("NDVI")
)

ndvi_2025 = (
    image_2025
    .normalizedDifference(["B8", "B4"])
    .rename("NDVI")
)

print("NDVI calculated successfully for 2019 and 2025!")

NDVI calculated successfully for 2019 and 2025!


In [14]:
# Step 9: Visualize NDVI Maps

ndvi_vis = {
    "min": -1,
    "max": 1,
    "palette": ["brown", "yellow", "green"]
}

ndvi_map = geemap.Map(center=[31.34, 74.24], zoom=11)

ndvi_map.addLayer(
    ndvi_2019,
    ndvi_vis,
    "NDVI 2019"
)

ndvi_map.addLayer(
    ndvi_2025,
    ndvi_vis,
    "NDVI 2025"
)

ndvi_map.addLayer(
    aoi,
    {"color": "black"},
    "AOI Boundary"
)

ndvi_map.addLayerControl()

ndvi_map

Map(center=[31.34, 74.24], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [15]:
# Step 10: Calculate NDBI for 2019 and 2025

ndbi_2019 = (
    image_2019
    .normalizedDifference(["B11", "B8"])
    .rename("NDBI")
)

ndbi_2025 = (
    image_2025
    .normalizedDifference(["B11", "B8"])
    .rename("NDBI")
)

print("NDBI calculated successfully for 2019 and 2025!")

NDBI calculated successfully for 2019 and 2025!


In [16]:
# Step 11: Visualize NDBI Maps

ndbi_vis = {
    "min": -1,
    "max": 1,
    "palette": ["blue", "white", "red"]
}

ndbi_map = geemap.Map(center=[31.34, 74.24], zoom=11)

ndbi_map.addLayer(
    ndbi_2019,
    ndbi_vis,
    "NDBI 2019"
)

ndbi_map.addLayer(
    ndbi_2025,
    ndbi_vis,
    "NDBI 2025"
)

ndbi_map.addLayer(
    aoi,
    {"color": "black"},
    "AOI Boundary"
)

ndbi_map.addLayerControl()

ndbi_map

Map(center=[31.34, 74.24], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [17]:
# Step 12: Land-Cover Classification

def classify_land(image):

    # Calculate indices
    ndvi = image.normalizedDifference(["B8", "B4"])
    ndbi = image.normalizedDifference(["B11", "B8"])

    # Class 0: Vegetation / Agriculture
    vegetation = ndvi.gt(0.3)

    # Class 1: Built-up / Urban
    built_up = ndbi.gt(0.1).And(ndvi.lt(0.3))

    # Class 2: Other
    other = vegetation.Or(built_up).Not()

    # Combine all three classes
    classification = (
        ee.Image(2)
        .where(vegetation, 0)
        .where(built_up, 1)
        .rename("Land_Class")
    )

    return classification.clip(aoi)


# Classify both years
classified_2019 = classify_land(image_2019)
classified_2025 = classify_land(image_2025)

print("Land-cover classification completed!")
print("Classes: Vegetation, Built-up, Other")

Land-cover classification completed!
Classes: Vegetation, Built-up, Other


In [18]:
# Step 13: Visualize Land-Cover Classification

class_vis = {
    "min": 0,
    "max": 2,
    "palette": ["green", "red", "yellow"]
}

class_map = geemap.Map(center=[31.34, 74.24], zoom=11)

class_map.addLayer(
    classified_2019,
    class_vis,
    "Land Classification 2019"
)

class_map.addLayer(
    classified_2025,
    class_vis,
    "Land Classification 2025"
)

class_map.addLayer(
    aoi,
    {"color": "black"},
    "AOI Boundary"
)

class_map.addLayerControl()

class_map

Map(center=[31.34, 74.24], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [19]:
# Step 14: Detect Vegetation-to-Built-up Change

# Vegetation in 2019
vegetation_2019 = classified_2019.eq(0)

# Built-up in 2025
builtup_2025 = classified_2025.eq(1)

# Detect possible conversion
change_map = (
    vegetation_2019
    .And(builtup_2025)
    .selfMask()
    .rename("Vegetation_to_Builtup")
)

print("Change detection completed successfully!")

Change detection completed successfully!


In [20]:
# Step 15: Visualize Change Detection Map

change_vis = {
    "palette": ["red"]
}

change_map_view = geemap.Map(
    center=[31.34, 74.24],
    zoom=11
)

# Add recent satellite imagery
change_map_view.addLayer(
    image_2025,
    rgb_vis,
    "Satellite Image 2025"
)

# Highlight possible vegetation-to-built-up conversion
change_map_view.addLayer(
    change_map,
    change_vis,
    "Possible Vegetation-to-Built-up Change"
)

# Add AOI boundary
change_map_view.addLayer(
    aoi,
    {"color": "yellow"},
    "AOI Boundary"
)

change_map_view.addLayerControl()

change_map_view

Map(center=[31.34, 74.24], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [21]:
# Step 16: Calculate Possible Converted Area

# Calculate pixel area in square meters
pixel_area = ee.Image.pixelArea()

changed_area = (
    pixel_area
    .updateMask(change_map)
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=10,
        maxPixels=1e10
    )
)

# Convert square meters into square kilometers
changed_km2 = (
    ee.Number(changed_area.get("area"))
    .divide(1_000_000)
    .getInfo()
)

# Calculate total AOI area
total_area_km2 = (
    aoi.area()
    .divide(1_000_000)
    .getInfo()
)

# Calculate affected percentage
change_percentage = (
    changed_km2 / total_area_km2
) * 100

print("AREA CHANGE STATISTICS")
print("----------------------")
print("Total AOI Area:", round(total_area_km2, 2), "km²")
print("Possible Converted Area:", round(changed_km2, 4), "km²")
print("Percentage Affected:", round(change_percentage, 2), "%")

AREA CHANGE STATISTICS
----------------------
Total AOI Area: 67.59 km²
Possible Converted Area: 1.9199 km²
Percentage Affected: 2.84 %


In [22]:
# Step 17: Calculate Land-Cover Area Statistics

class_names = {
    0: "Vegetation / Agriculture",
    1: "Built-up / Urban",
    2: "Other"
}

def calculate_class_area(classified_image, class_id):

    area_image = (
        ee.Image.pixelArea()
        .updateMask(classified_image.eq(class_id))
    )

    result = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=10,
        maxPixels=1e10
    )

    area_m2 = result.get("area").getInfo()

    if area_m2 is None:
        return 0.0

    return area_m2 / 1_000_000


# Calculate areas for both years
area_records = []

for class_id, class_name in class_names.items():

    area_2019 = calculate_class_area(
        classified_2019, class_id
    )

    area_2025 = calculate_class_area(
        classified_2025, class_id
    )

    area_records.append({
        "Land Class": class_name,
        "2019 Area (km²)": round(area_2019, 2),
        "2025 Area (km²)": round(area_2025, 2)
    })


# Create statistics table
area_df = pd.DataFrame(area_records)

display(area_df)

,Land Class,2019 Area (km²),2025 Area (km²)
0,Vegetation / Agriculture,28.44,30.56
1,Built-up / Urban,12.04,8.17
2,Other,27.05,28.79


In [23]:
# Step 18: Save Land-Cover Area Statistics

from pathlib import Path

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    (p for p in [current, *current.parents]
     if (p / "requirements.txt").exists()),
    current
)

output_folder = PROJECT_ROOT / "outputs"
output_folder.mkdir(parents=True, exist_ok=True)

csv_path = output_folder / "land_cover_area_statistics.csv"

area_df.to_csv(csv_path, index=False)

print("Area statistics saved successfully!")
print("File location:", csv_path)

Area statistics saved successfully!
File location: D:\Punjab-GeoGuardian\outputs\land_cover_area_statistics.csv


In [24]:
# Step 19: Generate Validation Points

validation_points = ee.FeatureCollection.randomPoints(
    region=aoi,
    points=70,
    seed=42
)

validation_samples = classified_2025.sampleRegions(
    collection=validation_points,
    scale=10,
    geometries=True
)

features = validation_samples.getInfo()["features"]

class_names = {
    0: "Vegetation",
    1: "Built-up",
    2: "Other"
}

validation_records = []

for feature in features:

    coordinates = feature["geometry"]["coordinates"]
    predicted_class = feature["properties"]["Land_Class"]

    validation_records.append({
        "Longitude": coordinates[0],
        "Latitude": coordinates[1],
        "Actual Class": "",
        "Predicted Class": class_names.get(
            predicted_class, "Unknown"
        ),
        "Correct/Incorrect": "",
        "Notes": ""
    })

# Keep 50 valid candidate points
validation_df = pd.DataFrame(validation_records).head(50)

print("Validation candidate points:", len(validation_df))

# Save CSV
validation_folder = PROJECT_ROOT / "data" / "validation_data"
validation_folder.mkdir(parents=True, exist_ok=True)

validation_path = validation_folder / "validation_points.csv"

validation_df.to_csv(validation_path, index=False)

print("Validation CSV saved successfully!")
print("File:", validation_path)

display(validation_df.head())

Validation candidate points: 50
Validation CSV saved successfully!
File: D:\Punjab-GeoGuardian\data\validation_data\validation_points.csv


,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,,Vegetation,,
1,74.248588,31.349092,,Other,,
2,74.225950,31.323490,,Vegetation,,
3,74.207265,31.334091,,Vegetation,,
4,74.225771,31.345230,,Other,,


In [25]:
# Step 20: Manually Inspect Validation Point 1

point_1_lon = validation_df.iloc[0]["Longitude"]
point_1_lat = validation_df.iloc[0]["Latitude"]

point_1_map = folium.Map(
    location=[point_1_lat, point_1_lon],
    zoom_start=17
)

# Add 2025 Sentinel-2 satellite imagery
tile_url = image_2025.getMapId(rgb_vis)["tile_fetcher"].url_format

folium.TileLayer(
    tiles=tile_url,
    attr="Google Earth Engine",
    name="Sentinel-2 2025",
    overlay=True,
    control=True
).add_to(point_1_map)

# Mark validation point
folium.Marker(
    [point_1_lat, point_1_lon],
    tooltip="Validation Point 1",
    icon=folium.Icon(color="red")
).add_to(point_1_map)

display(point_1_map)

In [26]:
display(validation_df.head())

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,,Vegetation,,
1,74.248588,31.349092,,Other,,
2,74.225950,31.323490,,Vegetation,,
3,74.207265,31.334091,,Vegetation,,
4,74.225771,31.345230,,Other,,


In [27]:
print("Total Validation Points:", len(validation_df))

display(validation_df)

Total Validation Points: 50


,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,,Vegetation,,
1,74.248588,31.349092,,Other,,
2,74.225950,31.323490,,Vegetation,,
3,74.207265,31.334091,,Vegetation,,
4,74.225771,31.345230,,Other,,
5,74.246162,31.306602,,Other,,
6,74.251912,31.354752,,Other,,
7,74.244905,31.348553,,Vegetation,,
8,74.274729,31.352326,,Other,,
9,74.248947,31.349811,,Vegetation,,


In [28]:
# Step 20: Display Validation Point 1 on Folium Map

point_1_lon = validation_df.iloc[0]["Longitude"]
point_1_lat = validation_df.iloc[0]["Latitude"]

point_1 = ee.Geometry.Point([
    point_1_lon,
    point_1_lat
])

point_map = geemap.Map(
    center=[point_1_lat, point_1_lon],
    zoom=17
)

point_map.addLayer(
    image_2025,
    rgb_vis,
    "Sentinel-2 Satellite 2025"
)

point_map.addLayer(
    point_1,
    {"color": "yellow"},
    "Validation Point 1"
)

display(point_map)

Map(center=[np.float64(31.322771737028916), np.float64(74.20205519390603)], controls=(WidgetControl(options=['…

In [29]:
# Step 22: Record Point 1 Manual Validation

validation_df.loc[0, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[0, "Predicted Class"]

validation_df.loc[0, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[0, "Notes"] = (
    "Agricultural fields and vegetation observed in satellite imagery."
)

# Save updated validation table
validation_df.to_csv(validation_path, index=False)

display(validation_df.head(1))

print("Point 1 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...


Point 1 validation saved successfully!


In [30]:
# Step 23: Inspect Validation Point 2

point_2_lon = validation_df.iloc[1]["Longitude"]
point_2_lat = validation_df.iloc[1]["Latitude"]

point_2 = ee.Geometry.Point([
    point_2_lon,
    point_2_lat
])

point_2_map = geemap.Map(
    center=[point_2_lat, point_2_lon],
    zoom=17
)

point_2_map.addLayer(
    image_2025,
    rgb_vis,
    "Sentinel-2 Satellite 2025"
)

point_2_map.addLayer(
    point_2,
    {"color": "yellow"},
    "Validation Point 2"
)

display(point_2_map)

Map(center=[np.float64(31.34909237485362), np.float64(74.24858792562343)], controls=(WidgetControl(options=['p…

In [31]:
# Inspect Point 2 using Google Satellite basemap

point_2_map = folium.Map(
    location=[point_2_lat, point_2_lon],
    zoom_start=18,
    tiles=None
)

folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google",
    name="Google Satellite",
    max_zoom=20
).add_to(point_2_map)

folium.Marker(
    [point_2_lat, point_2_lon],
    tooltip="Validation Point 2",
    icon=folium.Icon(color="red")
).add_to(point_2_map)

display(point_2_map)

In [32]:
# Step 24: Save Point 2 Manual Validation

validation_df.loc[1, "Actual Class"] = "Built-up"

predicted = validation_df.loc[1, "Predicted Class"]

validation_df.loc[1, "Correct/Incorrect"] = (
    "Correct" if predicted == "Built-up" else "Incorrect"
)

validation_df.loc[1, "Notes"] = (
    "Buildings observed using high-resolution satellite basemap."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[0:2])

print("Point 2 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...


Point 2 validation saved successfully!


In [37]:
# Step 28: Display Point 3 on Sentinel-2 Satellite Imagery

point_3_lon = validation_df.iloc[2]["Longitude"]
point_3_lat = validation_df.iloc[2]["Latitude"]

# Prepare satellite imagery from Earth Engine
satellite_image = image_2025.visualize(**rgb_vis)

# Get Earth Engine map tiles
map_info = satellite_image.getMapId()
tile_url = map_info["tile_fetcher"].url_format

# Create map
point_3_map = folium.Map(
    location=[point_3_lat, point_3_lon],
    zoom_start=17,
    tiles=None
)

# Add actual Sentinel-2 satellite imagery
folium.TileLayer(
    tiles=tile_url,
    attr="Google Earth Engine / Sentinel-2",
    name="Sentinel-2 Satellite 2025",
    overlay=True,
    control=True
).add_to(point_3_map)

# Add Point 3 marker
folium.Marker(
    location=[point_3_lat, point_3_lon],
    tooltip="Validation Point 3",
    icon=folium.Icon(color="red")
).add_to(point_3_map)

folium.LayerControl().add_to(point_3_map)

display(point_3_map)

In [38]:
print("POINT 1")
print("Latitude:", validation_df.iloc[0]["Latitude"])
print("Longitude:", validation_df.iloc[0]["Longitude"])

print("\nPOINT 3")
print("Latitude:", validation_df.iloc[2]["Latitude"])
print("Longitude:", validation_df.iloc[2]["Longitude"])

print("\nSatellite image bands:", image_2025.bandNames().getInfo())

POINT 1
Latitude: 31.322771737028916
Longitude: 74.20205519390603

POINT 3
Latitude: 31.32349038925621
Longitude: 74.22595038046362

Satellite image bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


In [39]:
# Create both validation points
point_1 = ee.Geometry.Point([
    validation_df.iloc[0]["Longitude"],
    validation_df.iloc[0]["Latitude"]
])

point_3 = ee.Geometry.Point([
    validation_df.iloc[2]["Longitude"],
    validation_df.iloc[2]["Latitude"]
])

# Create one map, using the same method as Point 1
comparison_map = geemap.Map(
    center=[31.3231, 74.2140],
    zoom=14
)

# Add 2025 satellite imagery
comparison_map.addLayer(
    image_2025,
    rgb_vis,
    "Sentinel-2 Satellite 2025"
)

# Add both markers
comparison_map.addLayer(
    point_1,
    {"color": "yellow"},
    "Point 1"
)

comparison_map.addLayer(
    point_3,
    {"color": "red"},
    "Point 3"
)

comparison_map.addLayerControl()

display(comparison_map)

Map(center=[31.3231, 74.214], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

In [40]:
# Step 31: Save Point 3 Manual Validation

validation_df.loc[2, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[2, "Predicted Class"]

validation_df.loc[2, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[2, "Notes"] = (
    "Vegetation and green agricultural fields observed in satellite imagery."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[0:3])

print("Point 3 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...


Point 3 validation saved successfully!


In [41]:
point_4_lon = validation_df.iloc[3]["Longitude"]
point_4_lat = validation_df.iloc[3]["Latitude"]

point_4 = ee.Geometry.Point([point_4_lon, point_4_lat])

point_4_map = geemap.Map(
    center=[point_4_lat, point_4_lon],
    zoom=17
)

point_4_map.addLayer(image_2025, rgb_vis, "Satellite 2025")
point_4_map.addLayer(point_4, {"color": "red"}, "Validation Point 4")
point_4_map.addLayerControl()

display(point_4_map)

Map(center=[np.float64(31.334090509608824), np.float64(74.20726542255393)], controls=(WidgetControl(options=['…

In [42]:
validation_df.loc[3, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[3, "Predicted Class"]

validation_df.loc[3, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[3, "Notes"] = (
    "Vegetation and agricultural fields observed in satellite imagery."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[0:4])

print("Point 4 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...
3,74.207265,31.334091,Vegetation,Vegetation,Correct,Vegetation and agricultural fields observed in...


Point 4 validation saved successfully!


In [43]:
point_5_lon = validation_df.iloc[4]["Longitude"]
point_5_lat = validation_df.iloc[4]["Latitude"]

point_5 = ee.Geometry.Point([point_5_lon, point_5_lat])

point_5_map = geemap.Map(
    center=[point_5_lat, point_5_lon],
    zoom=17
)

point_5_map.addLayer(image_2025, rgb_vis, "Satellite 2025")
point_5_map.addLayer(point_5, {"color": "red"}, "Validation Point 5")
point_5_map.addLayerControl()

display(point_5_map)

Map(center=[np.float64(31.345229619131906), np.float64(74.22577071740679)], controls=(WidgetControl(options=['…

In [44]:
validation_df.loc[4, "Actual Class"] = "Other"

predicted = validation_df.loc[4, "Predicted Class"]

validation_df.loc[4, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[4, "Notes"] = (
    "Brown bare soil/open land observed in satellite imagery."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[0:5])

print("Point 5 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...
3,74.207265,31.334091,Vegetation,Vegetation,Correct,Vegetation and agricultural fields observed in...
4,74.225771,31.345230,Other,Other,Correct,Brown bare soil/open land observed in satellit...


Point 5 validation saved successfully!


In [45]:
point_6_lon = validation_df.iloc[5]["Longitude"]
point_6_lat = validation_df.iloc[5]["Latitude"]

point_6 = ee.Geometry.Point([point_6_lon, point_6_lat])

point_6_map = geemap.Map(
    center=[point_6_lat, point_6_lon],
    zoom=17
)

point_6_map.addLayer(image_2025, rgb_vis, "Satellite 2025")
point_6_map.addLayer(point_6, {"color": "red"}, "Validation Point 6")
point_6_map.addLayerControl()

display(point_6_map)

Map(center=[np.float64(31.306602061914766), np.float64(74.24616247435631)], controls=(WidgetControl(options=['…

In [46]:
validation_df.loc[5, "Actual Class"] = "Other"

predicted = validation_df.loc[5, "Predicted Class"]

validation_df.loc[5, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[5, "Notes"] = (
    "Plots and brown bare soil observed in satellite imagery."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[0:6])

print("Point 6 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...
3,74.207265,31.334091,Vegetation,Vegetation,Correct,Vegetation and agricultural fields observed in...
4,74.225771,31.345230,Other,Other,Correct,Brown bare soil/open land observed in satellit...
5,74.246162,31.306602,Other,Other,Correct,Plots and brown bare soil observed in satellit...


Point 6 validation saved successfully!


In [47]:
point_7_lon = validation_df.iloc[6]["Longitude"]
point_7_lat = validation_df.iloc[6]["Latitude"]

point_7 = ee.Geometry.Point([point_7_lon, point_7_lat])

point_7_map = geemap.Map(
    center=[point_7_lat, point_7_lon],
    zoom=17
)

point_7_map.addLayer(image_2025, rgb_vis, "Satellite 2025")
point_7_map.addLayer(point_7, {"color": "red"}, "Validation Point 7")
point_7_map.addLayerControl()

display(point_7_map)

Map(center=[np.float64(31.35475176114357), np.float64(74.25191169217467)], controls=(WidgetControl(options=['p…

In [48]:
validation_df.loc[6, "Actual Class"] = "Other"

predicted = validation_df.loc[6, "Predicted Class"]

validation_df.loc[6, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[6, "Notes"] = (
    "Road surface observed; no clear surrounding built-up area."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[0:7])

print("Point 7 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...
3,74.207265,31.334091,Vegetation,Vegetation,Correct,Vegetation and agricultural fields observed in...
4,74.225771,31.345230,Other,Other,Correct,Brown bare soil/open land observed in satellit...
5,74.246162,31.306602,Other,Other,Correct,Plots and brown bare soil observed in satellit...
6,74.251912,31.354752,Other,Other,Correct,Road surface observed; no clear surrounding bu...


Point 7 validation saved successfully!


In [49]:
point_8_lon = validation_df.iloc[7]["Longitude"]
point_8_lat = validation_df.iloc[7]["Latitude"]

point_8 = ee.Geometry.Point([point_8_lon, point_8_lat])

point_8_map = geemap.Map(
    center=[point_8_lat, point_8_lon],
    zoom=17
)

point_8_map.addLayer(image_2025, rgb_vis, "Satellite 2025")
point_8_map.addLayer(point_8, {"color": "red"}, "Validation Point 8")
point_8_map.addLayerControl()

display(point_8_map)

Map(center=[np.float64(31.348553385683147), np.float64(74.24490483295854)], controls=(WidgetControl(options=['…

In [50]:
validation_df.loc[7, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[7, "Predicted Class"]

validation_df.loc[7, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[7, "Notes"] = (
    "Green vegetation observed around the validation point."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[0:8])

print("Point 8 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...
3,74.207265,31.334091,Vegetation,Vegetation,Correct,Vegetation and agricultural fields observed in...
4,74.225771,31.345230,Other,Other,Correct,Brown bare soil/open land observed in satellit...
5,74.246162,31.306602,Other,Other,Correct,Plots and brown bare soil observed in satellit...
6,74.251912,31.354752,Other,Other,Correct,Road surface observed; no clear surrounding bu...
7,74.244905,31.348553,Vegetation,Vegetation,Correct,Green vegetation observed around the validatio...


Point 8 validation saved successfully!


In [52]:
point_9_lon = validation_df.iloc[8]["Longitude"]
point_9_lat = validation_df.iloc[8]["Latitude"]

point_9 = ee.Geometry.Point([point_9_lon, point_9_lat])

point_9_map = geemap.Map(
    center=[point_9_lat, point_9_lon],
    zoom=17
)

point_9_map.addLayer(image_2025, rgb_vis, "Satellite 2025")
point_9_map.addLayer(point_9, {"color": "red"}, "Validation Point 9")
point_9_map.addLayerControl()

display(point_9_map)

Map(center=[np.float64(31.35232630987645), np.float64(74.27472890039131)], controls=(WidgetControl(options=['p…

In [53]:
validation_df.loc[8, "Actual Class"] = "Built-up"

predicted = validation_df.loc[8, "Predicted Class"]

validation_df.loc[8, "Correct/Incorrect"] = (
    "Correct" if predicted == "Built-up" else "Incorrect"
)

validation_df.loc[8, "Notes"] = (
    "House/building observed at the validation point."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[0:9])

print("Point 9 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...
3,74.207265,31.334091,Vegetation,Vegetation,Correct,Vegetation and agricultural fields observed in...
4,74.225771,31.345230,Other,Other,Correct,Brown bare soil/open land observed in satellit...
5,74.246162,31.306602,Other,Other,Correct,Plots and brown bare soil observed in satellit...
6,74.251912,31.354752,Other,Other,Correct,Road surface observed; no clear surrounding bu...
7,74.244905,31.348553,Vegetation,Vegetation,Correct,Green vegetation observed around the validatio...
8,74.274729,31.352326,Built-up,Other,Incorrect,House/building observed at the validation point.


Point 9 validation saved successfully!


In [54]:
point_10_lon = validation_df.iloc[9]["Longitude"]
point_10_lat = validation_df.iloc[9]["Latitude"]

point_10 = ee.Geometry.Point([point_10_lon, point_10_lat])

point_10_map = geemap.Map(
    center=[point_10_lat, point_10_lon],
    zoom=17
)

point_10_map.addLayer(image_2025, rgb_vis, "Satellite 2025")
point_10_map.addLayer(point_10, {"color": "red"}, "Validation Point 10")
point_10_map.addLayerControl()

display(point_10_map)

Map(center=[np.float64(31.349811027080914), np.float64(74.24894725173708)], controls=(WidgetControl(options=['…

In [55]:
validation_df.loc[9, "Actual Class"] = "Other"

predicted = validation_df.loc[9, "Predicted Class"]

validation_df.loc[9, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[9, "Notes"] = (
    "Road surface observed at the validation point."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[0:10])

print("Point 10 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...
3,74.207265,31.334091,Vegetation,Vegetation,Correct,Vegetation and agricultural fields observed in...
4,74.225771,31.345230,Other,Other,Correct,Brown bare soil/open land observed in satellit...
5,74.246162,31.306602,Other,Other,Correct,Plots and brown bare soil observed in satellit...
6,74.251912,31.354752,Other,Other,Correct,Road surface observed; no clear surrounding bu...
7,74.244905,31.348553,Vegetation,Vegetation,Correct,Green vegetation observed around the validatio...
8,74.274729,31.352326,Built-up,Other,Incorrect,House/building observed at the validation point.
9,74.248947,31.349811,Other,Vegetation,Incorrect,Road surface observed at the validation point.


Point 10 validation saved successfully!


In [56]:
point_11_lon = validation_df.iloc[10]["Longitude"]
point_11_lat = validation_df.iloc[10]["Latitude"]

point_11 = ee.Geometry.Point([point_11_lon, point_11_lat])

point_11_map = geemap.Map(
    center=[point_11_lat, point_11_lon],
    zoom=17
)

point_11_map.addLayer(image_2025, rgb_vis, "Satellite 2025")
point_11_map.addLayer(point_11, {"color": "red"}, "Validation Point 11")
point_11_map.addLayerControl()

display(point_11_map)

Map(center=[np.float64(31.34783473345585), np.float64(74.23547252247528)], controls=(WidgetControl(options=['p…

In [57]:
validation_df.loc[10, "Actual Class"] = "Built-up"

predicted = validation_df.loc[10, "Predicted Class"]

validation_df.loc[10, "Correct/Incorrect"] = (
    "Correct" if predicted == "Built-up" else "Incorrect"
)

validation_df.loc[10, "Notes"] = (
    "Houses and building structures observed in satellite imagery."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[0:11])

print("Point 11 validation saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...
3,74.207265,31.334091,Vegetation,Vegetation,Correct,Vegetation and agricultural fields observed in...
4,74.225771,31.345230,Other,Other,Correct,Brown bare soil/open land observed in satellit...
5,74.246162,31.306602,Other,Other,Correct,Plots and brown bare soil observed in satellit...
6,74.251912,31.354752,Other,Other,Correct,Road surface observed; no clear surrounding bu...
7,74.244905,31.348553,Vegetation,Vegetation,Correct,Green vegetation observed around the validatio...
8,74.274729,31.352326,Built-up,Other,Incorrect,House/building observed at the validation point.
9,74.248947,31.349811,Other,Vegetation,Incorrect,Road surface observed at the validation point.


Point 11 validation saved successfully!


In [58]:
# Check NDVI and NDBI at Point 11

point_11_values = ee.Image.cat([
    ndvi_2025.rename("NDVI"),
    ndbi_2025.rename("NDBI")
]).reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point_11,
    scale=10
).getInfo()

print("Point 11 NDVI:", point_11_values["NDVI"])
print("Point 11 NDBI:", point_11_values["NDBI"])
print("Model Prediction:", validation_df.loc[10, "Predicted Class"])
print("Actual Class:", validation_df.loc[10, "Actual Class"])

Point 11 NDVI: 0.31996179560649474
Point 11 NDBI: -0.07111024995155978
Model Prediction: Vegetation
Actual Class: Built-up


In [59]:
# Record Point 11 classification error

error_record = {
    "Point": 11,
    "Actual Class": "Built-up",
    "Predicted Class": "Vegetation",
    "NDVI": point_11_values["NDVI"],
    "NDBI": point_11_values["NDBI"],
    "Error Type": "Built-up misclassified as Vegetation",
    "Possible Reason": "NDVI threshold-based classification error"
}

error_df = pd.DataFrame([error_record])

display(error_df)

,Point,Actual Class,Predicted Class,NDVI,NDBI,Error Type,Possible Reason
0,11,Built-up,Vegetation,0.319962,-0.07111,Built-up misclassified as Vegetation,NDVI threshold-based classification error


In [60]:
error_analysis_path = PROJECT_ROOT / "outputs" / "error_analysis.csv"

error_df.to_csv(error_analysis_path, index=False)

print("Error analysis saved at:")
print(error_analysis_path)

Error analysis saved at:
D:\Punjab-GeoGuardian\outputs\error_analysis.csv


In [61]:
incorrect_points = validation_df[
    validation_df["Correct/Incorrect"] == "Incorrect"
]

display(incorrect_points[[
    "Actual Class",
    "Predicted Class",
    "Longitude",
    "Latitude",
    "Notes"
]])

print("Total incorrect points:", len(incorrect_points))

,Actual Class,Predicted Class,Longitude,Latitude,Notes
1,Built-up,Other,74.248588,31.349092,Buildings observed using high-resolution satel...
8,Built-up,Other,74.274729,31.352326,House/building observed at the validation point.
9,Other,Vegetation,74.248947,31.349811,Road surface observed at the validation point.
10,Built-up,Vegetation,74.235473,31.347835,Houses and building structures observed in sat...


Total incorrect points: 4


In [62]:
# Collect all incorrect validation points

incorrect_points = validation_df[
    validation_df["Correct/Incorrect"] == "Incorrect"
].copy()

error_df = pd.DataFrame({
    "Point": incorrect_points.index + 1,
    "Actual Class": incorrect_points["Actual Class"].values,
    "Predicted Class": incorrect_points["Predicted Class"].values,
    "Longitude": incorrect_points["Longitude"].values,
    "Latitude": incorrect_points["Latitude"].values,
    "Error Type": [
        f"{actual} misclassified as {predicted}"
        for actual, predicted in zip(
            incorrect_points["Actual Class"],
            incorrect_points["Predicted Class"]
        )
    ],
    "Possible Reason": [
        "Rule-based spectral threshold misclassification"
        for _ in range(len(incorrect_points))
    ]
})

error_analysis_path = PROJECT_ROOT / "outputs" / "error_analysis.csv"

error_df.to_csv(error_analysis_path, index=False)

display(error_df)

print("Total documented errors:", len(error_df))
print("Saved at:", error_analysis_path)

,Point,Actual Class,Predicted Class,Longitude,Latitude,Error Type,Possible Reason
0,2,Built-up,Other,74.248588,31.349092,Built-up misclassified as Other,Rule-based spectral threshold misclassification
1,9,Built-up,Other,74.274729,31.352326,Built-up misclassified as Other,Rule-based spectral threshold misclassification
2,10,Other,Vegetation,74.248947,31.349811,Other misclassified as Vegetation,Rule-based spectral threshold misclassification
3,11,Built-up,Vegetation,74.235473,31.347835,Built-up misclassified as Vegetation,Rule-based spectral threshold misclassification


Total documented errors: 4
Saved at: D:\Punjab-GeoGuardian\outputs\error_analysis.csv


In [63]:
point_12_lon = validation_df.iloc[11]["Longitude"]
point_12_lat = validation_df.iloc[11]["Latitude"]

point_12 = ee.Geometry.Point([point_12_lon, point_12_lat])

point_12_map = geemap.Map(
    center=[point_12_lat, point_12_lon],
    zoom=17
)

point_12_map.addLayer(
    image_2025, rgb_vis, "Satellite 2025"
)

point_12_map.addLayer(
    point_12, {"color": "red"}, "Validation Point 12"
)

point_12_map.addLayerControl()
display(point_12_map)

Map(center=[np.float64(31.345768608302375), np.float64(74.2544269749702)], controls=(WidgetControl(options=['p…

In [64]:
validation_df.loc[11, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[11, "Predicted Class"]

validation_df.loc[11, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[11, "Notes"] = (
    "Green area observed, but satellite imagery is blurry; "
    "class requires further verification."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[11]])

print("Point 12 saved as tentative Vegetation.")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
11,74.254427,31.345769,Vegetation,Other,Incorrect,"Green area observed, but satellite imagery is ..."


Point 12 saved as tentative Vegetation.


In [65]:
validation_df.loc[11, "Correct/Incorrect"] = "Needs Review"

validation_df.loc[11, "Notes"] = (
    "Possible vegetation observed, but imagery is blurry. "
    "Prediction mismatch requires further verification."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[11]])

print("Point 12 marked for further review.")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
11,74.254427,31.345769,Vegetation,Other,Needs Review,"Possible vegetation observed, but imagery is b..."


Point 12 marked for further review.


In [66]:
point_13_lon = validation_df.iloc[12]["Longitude"]
point_13_lat = validation_df.iloc[12]["Latitude"]

point_13 = ee.Geometry.Point([point_13_lon, point_13_lat])

point_13_map = geemap.Map(
    center=[point_13_lat, point_13_lon],
    zoom=17
)

point_13_map.addLayer(image_2025, rgb_vis, "Satellite 2025")
point_13_map.addLayer(point_13, {"color": "red"}, "Validation Point 13")
point_13_map.addLayerControl()

display(point_13_map)

Map(center=[np.float64(31.33444983572247), np.float64(74.21624857539513)], controls=(WidgetControl(options=['p…

In [67]:
validation_df.loc[12, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[12, "Predicted Class"]

validation_df.loc[12, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[12, "Notes"] = (
    "Green vegetation and agricultural area observed "
    "at the validation point."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[12]])

print("Point 13 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
12,74.216249,31.33445,Vegetation,Vegetation,Correct,Green vegetation and agricultural area observe...


Point 13 saved successfully!


In [68]:
point_14_lon = validation_df.iloc[13]["Longitude"]
point_14_lat = validation_df.iloc[13]["Latitude"]

point_14 = ee.Geometry.Point([point_14_lon, point_14_lat])

point_14_map = geemap.Map(
    center=[point_14_lat, point_14_lon],
    zoom=17
)

point_14_map.addLayer(image_2025, rgb_vis, "Satellite 2025")
point_14_map.addLayer(point_14, {"color": "red"}, "Validation Point 14")
point_14_map.addLayerControl()

display(point_14_map)

Map(center=[np.float64(31.306602061914766), np.float64(74.26682372589106)], controls=(WidgetControl(options=['…

In [69]:
validation_df.loc[13, "Actual Class"] = "Other"

predicted = validation_df.loc[13, "Predicted Class"]

validation_df.loc[13, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[13, "Notes"] = (
    "Brown bare soil/open land observed at the validation point."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[13]])

print("Point 14 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
13,74.266824,31.306602,Other,Built-up,Incorrect,Brown bare soil/open land observed at the vali...


Point 14 saved successfully!


In [71]:
point_15_lon = validation_df.iloc[14]["Longitude"]
point_15_lat = validation_df.iloc[14]["Latitude"]

point_15 = ee.Geometry.Point([
    point_15_lon,
    point_15_lat
])

point_15_map = geemap.Map(
    center=[point_15_lat, point_15_lon],
    zoom=18
)

point_15_map.add_basemap("Esri.WorldImagery")

point_15_map.addLayer(
    point_15,
    {"color": "red"},
    "Validation Point 15"
)

point_15_map.addLayerControl()

display(point_15_map)

Map(center=[np.float64(31.331305732228053), np.float64(74.2152604285826)], controls=(WidgetControl(options=['p…

In [72]:
validation_df.loc[14, "Actual Class"] = "Other"

predicted = validation_df.loc[14, "Predicted Class"]

validation_df.loc[14, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[14, "Notes"] = (
    "Rural open land with an unpaved dirt track observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[14]])

print("Point 15 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
14,74.21526,31.331306,Other,Other,Correct,Rural open land with an unpaved dirt track obs...


Point 15 saved successfully!


In [73]:
point_16_lon = validation_df.iloc[15]["Longitude"]
point_16_lat = validation_df.iloc[15]["Latitude"]

point_16 = ee.Geometry.Point([
    point_16_lon,
    point_16_lat
])

point_16_map = geemap.Map(
    center=[point_16_lat, point_16_lon],
    zoom=18
)

point_16_map.add_basemap("Esri.WorldImagery")

point_16_map.addLayer(
    point_16,
    {"color": "red"},
    "Validation Point 16"
)

point_16_map.addLayerControl()
display(point_16_map)

Map(center=[np.float64(31.346666923586497), np.float64(74.2304419568842)], controls=(WidgetControl(options=['p…

In [74]:
validation_df.loc[15, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[15, "Predicted Class"]

validation_df.loc[15, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[15, "Notes"] = (
    "Green grassland/agricultural field boundary adjacent "
    "to a residential area; mixed boundary observation."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[15]])

print("Point 16 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
15,74.230442,31.346667,Vegetation,Vegetation,Correct,Green grassland/agricultural field boundary ad...


Point 16 saved successfully!


In [75]:
point_17_lon = validation_df.iloc[16]["Longitude"]
point_17_lat = validation_df.iloc[16]["Latitude"]

point_17 = ee.Geometry.Point([
    point_17_lon,
    point_17_lat
])

point_17_map = geemap.Map(
    center=[point_17_lat, point_17_lon],
    zoom=18
)

point_17_map.add_basemap("Esri.WorldImagery")

point_17_map.addLayer(
    point_17,
    {"color": "red"},
    "Validation Point 17"
)

point_17_map.addLayerControl()
display(point_17_map)

Map(center=[np.float64(31.321693758687974), np.float64(74.27562721567543)], controls=(WidgetControl(options=['…

In [76]:
validation_df.loc[16, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[16, "Predicted Class"]

validation_df.loc[16, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[16, "Notes"] = (
    "Agricultural field boundary (field bund) observed "
    "between cultivated fields."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[16]])

print("Point 17 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
16,74.275627,31.321694,Vegetation,Built-up,Incorrect,Agricultural field boundary (field bund) obser...


Point 17 saved successfully!


In [77]:
point_18_lon = validation_df.iloc[17]["Longitude"]
point_18_lat = validation_df.iloc[17]["Latitude"]

point_18 = ee.Geometry.Point([
    point_18_lon,
    point_18_lat
])

point_18_map = geemap.Map(
    center=[point_18_lat, point_18_lon],
    zoom=18
)

point_18_map.add_basemap("Esri.WorldImagery")

point_18_map.addLayer(
    point_18,
    {"color": "red"},
    "Validation Point 18"
)

point_18_map.addLayerControl()
display(point_18_map)

Map(center=[np.float64(31.35834502228005), np.float64(74.22226728779873)], controls=(WidgetControl(options=['p…

In [78]:
validation_df.loc[17, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[17, "Predicted Class"]

validation_df.loc[17, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[17, "Notes"] = (
    "Grassland/open green space observed near a building."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[17]])

print("Point 18 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
17,74.222267,31.358345,Vegetation,Vegetation,Correct,Grassland/open green space observed near a bui...


Point 18 saved successfully!


In [79]:
point_19_lon = validation_df.iloc[18]["Longitude"]
point_19_lat = validation_df.iloc[18]["Latitude"]

point_19 = ee.Geometry.Point([
    point_19_lon,
    point_19_lat
])

point_19_map = geemap.Map(
    center=[point_19_lat, point_19_lon],
    zoom=18
)

point_19_map.add_basemap("Esri.WorldImagery")

point_19_map.addLayer(
    point_19,
    {"color": "red"},
    "Validation Point 19"
)

point_19_map.addLayerControl()
display(point_19_map)

Map(center=[np.float64(31.35888401145052), np.float64(74.27589671026067)], controls=(WidgetControl(options=['p…

In [80]:
validation_df.loc[18, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[18, "Predicted Class"]

validation_df.loc[18, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[18, "Notes"] = (
    "Tree line/hedgerow observed along an agricultural field boundary."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[18]])

print("Point 19 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
18,74.275897,31.358884,Vegetation,Vegetation,Correct,Tree line/hedgerow observed along an agricultu...


Point 19 saved successfully!


In [81]:
point_20_lon = validation_df.iloc[19]["Longitude"]
point_20_lat = validation_df.iloc[19]["Latitude"]

point_20 = ee.Geometry.Point([
    point_20_lon,
    point_20_lat
])

point_20_map = geemap.Map(
    center=[point_20_lat, point_20_lon],
    zoom=18
)

point_20_map.add_basemap("Esri.WorldImagery")

point_20_map.addLayer(
    point_20,
    {"color": "red"},
    "Validation Point 20"
)

point_20_map.addLayerControl()
display(point_20_map)

Map(center=[np.float64(31.319897128119734), np.float64(74.23385555496387)], controls=(WidgetControl(options=['…

In [82]:
validation_df.loc[19, "Actual Class"] = "Built-up"

predicted = validation_df.loc[19, "Predicted Class"]

validation_df.loc[19, "Correct/Incorrect"] = (
    "Correct" if predicted == "Built-up" else "Incorrect"
)

validation_df.loc[19, "Notes"] = (
    "Paved sidewalk or road-side urban infrastructure observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[19]])

print("Point 20 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
19,74.233856,31.319897,Built-up,Other,Incorrect,Paved sidewalk or road-side urban infrastructu...


Point 20 saved successfully!


In [83]:
point_21_lon = validation_df.iloc[20]["Longitude"]
point_21_lat = validation_df.iloc[20]["Latitude"]

point_21 = ee.Geometry.Point([
    point_21_lon,
    point_21_lat
])

point_21_map = geemap.Map(
    center=[point_21_lat, point_21_lon],
    zoom=18
)

point_21_map.add_basemap("Esri.WorldImagery")

point_21_map.addLayer(
    point_21,
    {"color": "red"},
    "Validation Point 21"
)

point_21_map.addLayerControl()
display(point_21_map)

Map(center=[np.float64(31.31100380680695), np.float64(74.25514562719749)], controls=(WidgetControl(options=['p…

In [84]:
validation_df.loc[20, "Actual Class"] = "Other"

predicted = validation_df.loc[20, "Predicted Class"]

validation_df.loc[20, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[20, "Notes"] = (
    "Location does not clearly show vegetation or built-up features; classified as other."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[20]])

print("Point 21 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
20,74.255146,31.311004,Other,Other,Correct,Location does not clearly show vegetation or b...


Point 21 saved successfully!


In [85]:
point_22_lon = validation_df.iloc[21]["Longitude"]
point_22_lat = validation_df.iloc[21]["Latitude"]

point_22 = ee.Geometry.Point([
    point_22_lon,
    point_22_lat
])

point_22_map = geemap.Map(
    center=[point_22_lat, point_22_lon],
    zoom=18
)

point_22_map.add_basemap("Esri.WorldImagery")

point_22_map.addLayer(
    point_22,
    {"color": "red"},
    "Validation Point 22"
)

point_22_map.addLayerControl()
display(point_22_map)

Map(center=[np.float64(31.367777332763303), np.float64(74.27176445995372)], controls=(WidgetControl(options=['…

In [86]:
validation_df.loc[21, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[21, "Predicted Class"]

validation_df.loc[21, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[21, "Notes"] = (
    "Agricultural field edge with trees or grass along the vegetation boundary."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[21]])

print("Point 22 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
21,74.271764,31.367777,Vegetation,Vegetation,Correct,Agricultural field edge with trees or grass al...


Point 22 saved successfully!


In [87]:
point_23_lon = validation_df.iloc[22]["Longitude"]
point_23_lat = validation_df.iloc[22]["Latitude"]

point_23 = ee.Geometry.Point([
    point_23_lon,
    point_23_lat
])

point_23_map = geemap.Map(
    center=[point_23_lat, point_23_lon],
    zoom=18
)

point_23_map.add_basemap("Esri.WorldImagery")

point_23_map.addLayer(
    point_23,
    {"color": "red"},
    "Validation Point 23"
)

point_23_map.addLayerControl()
display(point_23_map)

Map(center=[np.float64(31.30121217021005), np.float64(74.26682372589106)], controls=(WidgetControl(options=['p…

In [88]:
validation_df.loc[22, "Actual Class"] = "Other"

predicted = validation_df.loc[22, "Predicted Class"]

validation_df.loc[22, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[22, "Notes"] = (
    "Rural dirt path or agricultural field track between cultivated fields."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[22]])

print("Point 23 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
22,74.266824,31.301212,Other,Other,Correct,Rural dirt path or agricultural field track be...


Point 23 saved successfully!


In [89]:
point_24_lon = validation_df.iloc[23]["Longitude"]
point_24_lat = validation_df.iloc[23]["Latitude"]

point_24 = ee.Geometry.Point([
    point_24_lon,
    point_24_lat
])

point_24_map = geemap.Map(
    center=[point_24_lat, point_24_lon],
    zoom=18
)

point_24_map.add_basemap("Esri.WorldImagery")

point_24_map.addLayer(
    point_24,
    {"color": "red"},
    "Validation Point 24"
)

point_24_map.addLayerControl()
display(point_24_map)

Map(center=[np.float64(31.35780603310958), np.float64(74.27868148764144)], controls=(WidgetControl(options=['p…

In [90]:
validation_df.loc[23, "Actual Class"] = "Built-up"

predicted = validation_df.loc[23, "Predicted Class"]

validation_df.loc[23, "Correct/Incorrect"] = (
    "Correct" if predicted == "Built-up" else "Incorrect"
)

validation_df.loc[23, "Notes"] = (
    "Building roof or permanent constructed structure observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[23]])

print("Point 24 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
23,74.278681,31.357806,Built-up,Other,Incorrect,Building roof or permanent constructed structu...


Point 24 saved successfully!


In [91]:
validation_df.loc[23, "Actual Class"] = "Built-up"
validation_df.loc[23, "Correct/Incorrect"] = "Incorrect"

validation_df.loc[23, "Notes"] = (
    "Building roof observed, but the model predicted a different class."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[23]])

print("Point 24 incorrect prediction recorded!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
23,74.278681,31.357806,Built-up,Other,Incorrect,"Building roof observed, but the model predicte..."


Point 24 incorrect prediction recorded!


In [92]:
point_25_lon = validation_df.iloc[24]["Longitude"]
point_25_lat = validation_df.iloc[24]["Latitude"]

point_25 = ee.Geometry.Point([
    point_25_lon,
    point_25_lat
])

point_25_map = geemap.Map(
    center=[point_25_lat, point_25_lon],
    zoom=18
)

point_25_map.add_basemap("Esri.WorldImagery")

point_25_map.addLayer(
    point_25,
    {"color": "red"},
    "Validation Point 25"
)

point_25_map.addLayerControl()
display(point_25_map)

Map(center=[np.float64(31.3767604856045), np.float64(74.20951121076423)], controls=(WidgetControl(options=['po…

In [93]:
validation_df.loc[24, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[24, "Predicted Class"]

validation_df.loc[24, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[24, "Notes"] = (
    "Green vegetation or agricultural field observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[24]])

print("Point 25 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
24,74.209511,31.37676,Vegetation,Vegetation,Correct,Green vegetation or agricultural field observed.


Point 25 saved successfully!


In [94]:
point_26_lon = validation_df.iloc[25]["Longitude"]
point_26_lat = validation_df.iloc[25]["Latitude"]

point_26 = ee.Geometry.Point([
    point_26_lon,
    point_26_lat
])

point_26_map = geemap.Map(
    center=[point_26_lat, point_26_lon],
    zoom=18
)

point_26_map.add_basemap("Esri.WorldImagery")

point_26_map.addLayer(
    point_26,
    {"color": "red"},
    "Validation Point 26"
)

point_26_map.addLayerControl()
display(point_26_map)

Map(center=[np.float64(31.341905852580663), np.float64(74.27311193287989)], controls=(WidgetControl(options=['…

In [95]:
validation_df.loc[25, "Actual Class"] = "Other"

predicted = validation_df.loc[25, "Predicted Class"]

validation_df.loc[25, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[25, "Notes"] = (
    "Non-vegetated area without clearly identifiable built-up structures."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[25]])

print("Point 26 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
25,74.273112,31.341906,Other,Other,Correct,Non-vegetated area without clearly identifiabl...


Point 26 saved successfully!


In [96]:
point_27_lon = validation_df.iloc[26]["Longitude"]
point_27_lat = validation_df.iloc[26]["Latitude"]

point_27 = ee.Geometry.Point([
    point_27_lon,
    point_27_lat
])

point_27_map = geemap.Map(
    center=[point_27_lat, point_27_lon],
    zoom=18
)

point_27_map.add_basemap("Esri.WorldImagery")

point_27_map.addLayer(
    point_27,
    {"color": "red"},
    "Validation Point 27"
)

point_27_map.addLayerControl()
display(point_27_map)

Map(center=[np.float64(31.33894141214307), np.float64(74.2596372036181)], controls=(WidgetControl(options=['po…

In [97]:
validation_df.loc[26, "Actual Class"] = "Other"

predicted = validation_df.loc[26, "Predicted Class"]

validation_df.loc[26, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[26, "Notes"] = (
    "Area does not clearly show vegetation or built-up structures."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[26]])

print("Point 27 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
26,74.259637,31.338941,Other,Other,Correct,Area does not clearly show vegetation or built...


Point 27 saved successfully!


In [98]:
point_28_lon = validation_df.iloc[27]["Longitude"]
point_28_lat = validation_df.iloc[27]["Latitude"]

point_28 = ee.Geometry.Point([
    point_28_lon,
    point_28_lat
])

point_28_map = geemap.Map(
    center=[point_28_lat, point_28_lon],
    zoom=18
)

point_28_map.add_basemap("Esri.WorldImagery")

point_28_map.addLayer(
    point_28,
    {"color": "red"},
    "Validation Point 28"
)

point_28_map.addLayerControl()
display(point_28_map)

Map(center=[np.float64(31.344510966904608), np.float64(74.20430098211634)], controls=(WidgetControl(options=['…

In [100]:
validation_df.loc[27, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[27, "Predicted Class"]

validation_df.loc[27, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[27, "Notes"] = (
    "Green vegetation or agricultural field observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[27]])

print("Point 28 updated to Vegetation successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
27,74.204301,31.344511,Vegetation,Vegetation,Correct,Green vegetation or agricultural field observed.


Point 28 updated to Vegetation successfully!


In [101]:
point_29_lon = validation_df.iloc[28]["Longitude"]
point_29_lat = validation_df.iloc[28]["Latitude"]

point_29 = ee.Geometry.Point([
    point_29_lon,
    point_29_lat
])

point_29_map = geemap.Map(
    center=[point_29_lat, point_29_lon],
    zoom=18
)

point_29_map.add_basemap("Esri.WorldImagery")

point_29_map.addLayer(
    point_29,
    {"color": "red"},
    "Validation Point 29"
)

point_29_map.addLayerControl()
display(point_29_map)

Map(center=[np.float64(31.35088900542186), np.float64(74.27607637331748)], controls=(WidgetControl(options=['p…

In [103]:
validation_df.loc[28, "Actual Class"] = "Built-up"

predicted = validation_df.loc[28, "Predicted Class"]

validation_df.loc[28, "Correct/Incorrect"] = (
    "Correct" if predicted == "Built-up" else "Incorrect"
)

validation_df.loc[28, "Notes"] = (
    "Building or urban infrastructure observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[28]])

print("Point 29 updated to Built-up successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
28,74.276076,31.350889,Built-up,Built-up,Correct,Building or urban infrastructure observed.


Point 29 updated to Built-up successfully!


In [104]:
point_30_lon = validation_df.iloc[29]["Longitude"]
point_30_lat = validation_df.iloc[29]["Latitude"]

point_30 = ee.Geometry.Point([
    point_30_lon,
    point_30_lat
])

point_30_map = geemap.Map(
    center=[point_30_lat, point_30_lon],
    zoom=18
)

point_30_map.add_basemap("Esri.WorldImagery")

point_30_map.addLayer(
    point_30,
    {"color": "red"},
    "Validation Point 30"
)

point_30_map.addLayerControl()
display(point_30_map)

Map(center=[np.float64(31.359602663677816), np.float64(74.27850182458461)], controls=(WidgetControl(options=['…

In [105]:
validation_df.loc[29, "Actual Class"] = "Built-up"

predicted = validation_df.loc[29, "Predicted Class"]

validation_df.loc[29, "Correct/Incorrect"] = (
    "Correct" if predicted == "Built-up" else "Incorrect"
)

validation_df.loc[29, "Notes"] = (
    "Building or urban infrastructure observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[29]])

print("Point 30 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
29,74.278502,31.359603,Built-up,Other,Incorrect,Building or urban infrastructure observed.


Point 30 saved successfully!


In [106]:
point_31_lon = validation_df.iloc[30]["Longitude"]
point_31_lat = validation_df.iloc[30]["Latitude"]

point_31 = ee.Geometry.Point([point_31_lon, point_31_lat])

point_31_map = geemap.Map(
    center=[point_31_lat, point_31_lon],
    zoom=18
)

point_31_map.add_basemap("Esri.WorldImagery")
point_31_map.addLayer(
    point_31,
    {"color": "red"},
    "Validation Point 31"
)

point_31_map.addLayerControl()
display(point_31_map)

Map(center=[np.float64(31.323310726199388), np.float64(74.23933527819699)], controls=(WidgetControl(options=['…

In [107]:
validation_df.loc[30, "Actual Class"] = "Built-up"

predicted = validation_df.loc[30, "Predicted Class"]

validation_df.loc[30, "Correct/Incorrect"] = (
    "Correct" if predicted == "Built-up" else "Incorrect"
)

validation_df.loc[30, "Notes"] = (
    "Built-up area observed at the validation point."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[30]])

print("Point 31 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
30,74.239335,31.323311,Built-up,Other,Incorrect,Built-up area observed at the validation point.


Point 31 saved successfully!


In [108]:
point_32_lon = validation_df.iloc[31]["Longitude"]
point_32_lat = validation_df.iloc[31]["Latitude"]

point_32 = ee.Geometry.Point([point_32_lon, point_32_lat])

point_32_map = geemap.Map(
    center=[point_32_lat, point_32_lon],
    zoom=18
)

point_32_map.add_basemap("Esri.WorldImagery")
point_32_map.addLayer(
    point_32,
    {"color": "red"},
    "Validation Point 32"
)

point_32_map.addLayerControl()
display(point_32_map)

Map(center=[np.float64(31.338671917557832), np.float64(74.2626914755841)], controls=(WidgetControl(options=['p…

In [109]:
validation_df.loc[31, "Actual Class"] = "Other"

predicted = validation_df.loc[31, "Predicted Class"]

validation_df.loc[31, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[31, "Notes"] = "Bare soil or open land observed."

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[31]])

print("Point 32 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
31,74.262691,31.338672,Other,Other,Correct,Bare soil or open land observed.


Point 32 saved successfully!


In [110]:
point_33_lon = validation_df.iloc[32]["Longitude"]
point_33_lat = validation_df.iloc[32]["Latitude"]

point_33 = ee.Geometry.Point([point_33_lon, point_33_lat])

point_33_map = geemap.Map(
    center=[point_33_lat, point_33_lon],
    zoom=18
)

point_33_map.add_basemap("Esri.WorldImagery")
point_33_map.addLayer(
    point_33,
    {"color": "red"},
    "Validation Point 33"
)

point_33_map.addLayerControl()
display(point_33_map)

Map(center=[np.float64(31.32483786218239), np.float64(74.21130784133247)], controls=(WidgetControl(options=['p…

In [112]:
validation_df.loc[32, "Actual Class"] = "Other"

predicted = validation_df.loc[32, "Predicted Class"]

validation_df.loc[32, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[32, "Notes"] = (
    "Open gap between houses with mixed or unclear land cover."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[32]])

print("Point 33 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
32,74.211308,31.324838,Other,Other,Correct,Open gap between houses with mixed or unclear ...


Point 33 saved successfully!


In [113]:
point_34_lon = validation_df.iloc[33]["Longitude"]
point_34_lat = validation_df.iloc[33]["Latitude"]

point_34 = ee.Geometry.Point([point_34_lon, point_34_lat])

point_34_map = geemap.Map(
    center=[point_34_lat, point_34_lon],
    zoom=18
)

point_34_map.add_basemap("Esri.WorldImagery")
point_34_map.addLayer(
    point_34,
    {"color": "red"},
    "Validation Point 34"
)

point_34_map.addLayerControl()
display(point_34_map)

Map(center=[np.float64(31.35636872865499), np.float64(74.26969833480024)], controls=(WidgetControl(options=['p…

In [114]:
validation_df.loc[33, "Actual Class"] = "Other"

predicted = validation_df.loc[33, "Predicted Class"]

validation_df.loc[33, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[33, "Notes"] = (
    "Bare soil or mixed open land observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[33]])

print("Point 34 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
33,74.269698,31.356369,Other,Vegetation,Incorrect,Bare soil or mixed open land observed.


Point 34 saved successfully!


In [115]:
point_35_lon = validation_df.iloc[34]["Longitude"]
point_35_lat = validation_df.iloc[34]["Latitude"]

point_35 = ee.Geometry.Point([point_35_lon, point_35_lat])

point_35_map = geemap.Map(
    center=[point_35_lat, point_35_lon],
    zoom=18
)

point_35_map.add_basemap("Esri.WorldImagery")
point_35_map.addLayer(
    point_35,
    {"color": "red"},
    "Validation Point 35"
)

point_35_map.addLayerControl()
display(point_35_map)

Map(center=[np.float64(31.34415164079096), np.float64(74.27041698702753)], controls=(WidgetControl(options=['p…

In [116]:
validation_df.loc[34, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[34, "Predicted Class"]

validation_df.loc[34, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[34, "Notes"] = (
    "Green vegetation or agricultural cover observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[34]])

print("Point 35 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
34,74.270417,31.344152,Vegetation,Vegetation,Correct,Green vegetation or agricultural cover observed.


Point 35 saved successfully!


In [117]:
point_36_lon = validation_df.iloc[35]["Longitude"]
point_36_lat = validation_df.iloc[35]["Latitude"]

point_36 = ee.Geometry.Point([point_36_lon, point_36_lat])

point_36_map = geemap.Map(
    center=[point_36_lat, point_36_lon],
    zoom=18
)

point_36_map.add_basemap("Esri.WorldImagery")
point_36_map.addLayer(
    point_36,
    {"color": "red"},
    "Validation Point 36"
)

point_36_map.addLayerControl()
display(point_36_map)

Map(center=[np.float64(31.357626370052753), np.float64(74.21319430342912)], controls=(WidgetControl(options=['…

In [118]:
validation_df.loc[35, "Actual Class"] = "Other"

predicted = validation_df.loc[35, "Predicted Class"]

validation_df.loc[35, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[35, "Notes"] = (
    "Open land or mixed land cover observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[35]])

print("Point 36 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
35,74.213194,31.357626,Other,Other,Correct,Open land or mixed land cover observed.


Point 36 saved successfully!


In [119]:
point_37_lon = validation_df.iloc[36]["Longitude"]
point_37_lat = validation_df.iloc[36]["Latitude"]

point_37 = ee.Geometry.Point([point_37_lon, point_37_lat])

point_37_map = geemap.Map(
    center=[point_37_lat, point_37_lon],
    zoom=18
)

point_37_map.add_basemap("Esri.WorldImagery")
point_37_map.addLayer(
    point_37,
    {"color": "red"},
    "Validation Point 37"
)

point_37_map.addLayerControl()
display(point_37_map)

Map(center=[np.float64(31.354482266558335), np.float64(74.21184683050294)], controls=(WidgetControl(options=['…

In [120]:
validation_df.loc[36, "Actual Class"] = "Other"

predicted = validation_df.loc[36, "Predicted Class"]

validation_df.loc[36, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[36, "Notes"] = (
    "Open or bare land observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[36]])

print("Point 37 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
36,74.211847,31.354482,Other,Other,Correct,Open or bare land observed.


Point 37 saved successfully!


In [121]:
point_38_lon = validation_df.iloc[37]["Longitude"]
point_38_lat = validation_df.iloc[37]["Latitude"]

point_38 = ee.Geometry.Point([point_38_lon, point_38_lat])

point_38_map = geemap.Map(
    center=[point_38_lat, point_38_lon],
    zoom=18
)

point_38_map.add_basemap("Esri.WorldImagery")
point_38_map.addLayer(
    point_38,
    {"color": "red"},
    "Validation Point 38"
)

point_38_map.addLayerControl()
display(point_38_map)

Map(center=[np.float64(31.319268307420852), np.float64(74.27374075357878)], controls=(WidgetControl(options=['…

In [122]:
validation_df.loc[37, "Actual Class"] = "Other"

predicted = validation_df.loc[37, "Predicted Class"]

validation_df.loc[37, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[37, "Notes"] = (
    "Open or bare land observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[37]])

print("Point 38 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
37,74.273741,31.319268,Other,Other,Correct,Open or bare land observed.


Point 38 saved successfully!


In [123]:
point_39_lon = validation_df.iloc[38]["Longitude"]
point_39_lat = validation_df.iloc[38]["Latitude"]

point_39 = ee.Geometry.Point([point_39_lon, point_39_lat])

point_39_map = geemap.Map(
    center=[point_39_lat, point_39_lon],
    zoom=18
)

point_39_map.add_basemap("Esri.WorldImagery")
point_39_map.addLayer(
    point_39,
    {"color": "red"},
    "Validation Point 39"
)

point_39_map.addLayerControl()
display(point_39_map)

Map(center=[np.float64(31.371909583070256), np.float64(74.2626914755841)], controls=(WidgetControl(options=['p…

In [124]:
validation_df.loc[38, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[38, "Predicted Class"]

validation_df.loc[38, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[38, "Notes"] = (
    "Green vegetation or agricultural cover observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[38]])

print("Point 39 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
38,74.262691,31.37191,Vegetation,Vegetation,Correct,Green vegetation or agricultural cover observed.


Point 39 saved successfully!


In [125]:
point_40_lon = validation_df.iloc[39]["Longitude"]
point_40_lat = validation_df.iloc[39]["Latitude"]

point_40 = ee.Geometry.Point([point_40_lon, point_40_lat])

point_40_map = geemap.Map(
    center=[point_40_lat, point_40_lon],
    zoom=18
)

point_40_map.add_basemap("Esri.WorldImagery")
point_40_map.addLayer(
    point_40,
    {"color": "red"},
    "Validation Point 40"
)

point_40_map.addLayerControl()
display(point_40_map)

Map(center=[np.float64(31.371101099314547), np.float64(74.25846939374874)], controls=(WidgetControl(options=['…

In [126]:
validation_df.loc[39, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[39, "Predicted Class"]

validation_df.loc[39, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[39, "Notes"] = (
    "Green vegetation or agricultural cover observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[39]])

print("Point 40 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
39,74.258469,31.371101,Vegetation,Vegetation,Correct,Green vegetation or agricultural cover observed.


Point 40 saved successfully!


In [127]:
point_41_lon = validation_df.iloc[40]["Longitude"]
point_41_lat = validation_df.iloc[40]["Latitude"]

point_41 = ee.Geometry.Point([point_41_lon, point_41_lat])

point_41_map = geemap.Map(
    center=[point_41_lat, point_41_lon],
    zoom=18
)

point_41_map.add_basemap("Esri.WorldImagery")
point_41_map.addLayer(
    point_41,
    {"color": "red"},
    "Validation Point 41"
)

point_41_map.addLayerControl()
display(point_41_map)


Map(center=[np.float64(31.357716201581166), np.float64(74.267901704232)], controls=(WidgetControl(options=['po…

In [128]:
validation_df.loc[40, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[40, "Predicted Class"]

validation_df.loc[40, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[40, "Notes"] = (
    "Green vegetation or agricultural cover observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[40]])

print("Point 41 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
40,74.267902,31.357716,Vegetation,Vegetation,Correct,Green vegetation or agricultural cover observed.


Point 41 saved successfully!


In [129]:
point_42_lon = validation_df.iloc[41]["Longitude"]
point_42_lat = validation_df.iloc[41]["Latitude"]

point_42 = ee.Geometry.Point([point_42_lon, point_42_lat])

point_42_map = geemap.Map(
    center=[point_42_lat, point_42_lon],
    zoom=18
)

point_42_map.add_basemap("Esri.WorldImagery")
point_42_map.addLayer(
    point_42,
    {"color": "red"},
    "Validation Point 42"
)

point_42_map.addLayerControl()
display(point_42_map)

Map(center=[np.float64(31.30040368645434), np.float64(74.25433714344179)], controls=(WidgetControl(options=['p…

In [130]:
validation_df.loc[41, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[41, "Predicted Class"]

validation_df.loc[41, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[41, "Notes"] = (
    "Green vegetation or agricultural cover observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[41]])

print("Point 42 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
41,74.254337,31.300404,Vegetation,Vegetation,Correct,Green vegetation or agricultural cover observed.


Point 42 saved successfully!


In [131]:
point_43_lon = validation_df.iloc[42]["Longitude"]
point_43_lat = validation_df.iloc[42]["Latitude"]

point_43 = ee.Geometry.Point([point_43_lon, point_43_lat])

point_43_map = geemap.Map(
    center=[point_43_lat, point_43_lon],
    zoom=18
)

point_43_map.add_basemap("Esri.WorldImagery")
point_43_map.addLayer(
    point_43,
    {"color": "red"},
    "Validation Point 43"
)

point_43_map.addLayerControl()
display(point_43_map)

Map(center=[np.float64(31.313159763488837), np.float64(74.20421115058792)], controls=(WidgetControl(options=['…

In [132]:
validation_df.loc[42, "Actual Class"] = "Other"

predicted = validation_df.loc[42, "Predicted Class"]

validation_df.loc[42, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[42, "Notes"] = (
    "Open or bare land observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[42]])

print("Point 43 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
42,74.204211,31.31316,Other,Other,Correct,Open or bare land observed.


Point 43 saved successfully!


In [133]:
point_44_lon = validation_df.iloc[43]["Longitude"]
point_44_lat = validation_df.iloc[43]["Latitude"]

point_44 = ee.Geometry.Point([point_44_lon, point_44_lat])

point_44_map = geemap.Map(
    center=[point_44_lat, point_44_lon],
    zoom=18
)

point_44_map.add_basemap("Esri.WorldImagery")
point_44_map.addLayer(
    point_44,
    {"color": "red"},
    "Validation Point 44"
)

point_44_map.addLayerControl()
display(point_44_map)

Map(center=[np.float64(31.339300738256718), np.float64(74.23358606037863)], controls=(WidgetControl(options=['…

In [134]:
validation_df.loc[43, "Actual Class"] = "Built-up"

predicted = validation_df.loc[43, "Predicted Class"]

validation_df.loc[43, "Correct/Incorrect"] = (
    "Correct" if predicted == "Built-up" else "Incorrect"
)

validation_df.loc[43, "Notes"] = (
    "Building or urban infrastructure observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[43]])

print("Point 44 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
43,74.233586,31.339301,Built-up,Built-up,Correct,Building or urban infrastructure observed.


Point 44 saved successfully!


In [135]:
point_45_lon = validation_df.iloc[44]["Longitude"]
point_45_lat = validation_df.iloc[44]["Latitude"]

point_45 = ee.Geometry.Point([point_45_lon, point_45_lat])

point_45_map = geemap.Map(
    center=[point_45_lat, point_45_lon],
    zoom=18
)

point_45_map.add_basemap("Esri.WorldImagery")
point_45_map.addLayer(
    point_45,
    {"color": "red"},
    "Validation Point 45"
)

point_45_map.addLayerControl()
display(point_45_map)

Map(center=[np.float64(31.303278295363523), np.float64(74.21786554290654)], controls=(WidgetControl(options=['…

In [136]:
validation_df.loc[44, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[44, "Predicted Class"]

validation_df.loc[44, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[44, "Notes"] = (
    "Green vegetation or agricultural cover observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[44]])

print("Point 45 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
44,74.217866,31.303278,Vegetation,Vegetation,Correct,Green vegetation or agricultural cover observed.


Point 45 saved successfully!


In [137]:
point_46_lon = validation_df.iloc[45]["Longitude"]
point_46_lat = validation_df.iloc[45]["Latitude"]

point_46 = ee.Geometry.Point([point_46_lon, point_46_lat])

point_46_map = geemap.Map(
    center=[point_46_lat, point_46_lon],
    zoom=18
)

point_46_map.add_basemap("Esri.WorldImagery")
point_46_map.addLayer(
    point_46,
    {"color": "red"},
    "Validation Point 46"
)

point_46_map.addLayerControl()
display(point_46_map)

Map(center=[np.float64(31.329239607074577), np.float64(74.21948251041796)], controls=(WidgetControl(options=['…

In [138]:
validation_df.loc[45, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[45, "Predicted Class"]

validation_df.loc[45, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[45, "Notes"] = (
    "Green vegetation or agricultural cover observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[45]])

print("Point 46 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
45,74.219483,31.32924,Vegetation,Vegetation,Correct,Green vegetation or agricultural cover observed.


Point 46 saved successfully!


In [139]:
point_47_lon = validation_df.iloc[46]["Longitude"]
point_47_lat = validation_df.iloc[46]["Latitude"]

point_47 = ee.Geometry.Point([point_47_lon, point_47_lat])

point_47_map = geemap.Map(
    center=[point_47_lat, point_47_lon],
    zoom=18
)

point_47_map.add_basemap("Esri.WorldImagery")
point_47_map.addLayer(
    point_47,
    {"color": "red"},
    "Validation Point 47"
)

point_47_map.addLayerControl()
display(point_47_map)

Map(center=[np.float64(31.333192194324702), np.float64(74.26583557907853)], controls=(WidgetControl(options=['…

In [140]:
validation_df.loc[46, "Actual Class"] = "Built-up"

predicted = validation_df.loc[46, "Predicted Class"]

validation_df.loc[46, "Correct/Incorrect"] = (
    "Correct" if predicted == "Built-up" else "Incorrect"
)

validation_df.loc[46, "Notes"] = (
    "Building or urban infrastructure observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[46]])

print("Point 47 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
46,74.265836,31.333192,Built-up,Built-up,Correct,Building or urban infrastructure observed.


Point 47 saved successfully!


In [141]:
point_48_lon = validation_df.iloc[47]["Longitude"]
point_48_lat = validation_df.iloc[47]["Latitude"]

point_48 = ee.Geometry.Point([point_48_lon, point_48_lat])

point_48_map = geemap.Map(
    center=[point_48_lat, point_48_lon],
    zoom=18
)

point_48_map.add_basemap("Esri.WorldImagery")
point_48_map.addLayer(
    point_48,
    {"color": "red"},
    "Validation Point 48"
)

point_48_map.addLayerControl()
display(point_48_map)

Map(center=[np.float64(31.346217765944438), np.float64(74.2431082023903)], controls=(WidgetControl(options=['p…

In [142]:
validation_df.loc[47, "Actual Class"] = "Other"

predicted = validation_df.loc[47, "Predicted Class"]

validation_df.loc[47, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[47, "Notes"] = (
    "Open or bare land observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[47]])

print("Point 48 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
47,74.243108,31.346218,Other,Other,Correct,Open or bare land observed.


Point 48 saved successfully!


In [143]:
point_49_lon = validation_df.iloc[48]["Longitude"]
point_49_lat = validation_df.iloc[48]["Latitude"]

point_49 = ee.Geometry.Point([point_49_lon, point_49_lat])

point_49_map = geemap.Map(
    center=[point_49_lat, point_49_lon],
    zoom=18
)

point_49_map.add_basemap("Esri.WorldImagery")
point_49_map.addLayer(
    point_49,
    {"color": "red"},
    "Validation Point 49"
)

point_49_map.addLayerControl()
display(point_49_map)

Map(center=[np.float64(31.32160392715956), np.float64(74.23735898457193)], controls=(WidgetControl(options=['p…

In [144]:
validation_df.loc[48, "Actual Class"] = "Other"

predicted = validation_df.loc[48, "Predicted Class"]

validation_df.loc[48, "Correct/Incorrect"] = (
    "Correct" if predicted == "Other" else "Incorrect"
)

validation_df.loc[48, "Notes"] = (
    "Open or bare land observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[48]])

print("Point 49 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
48,74.237359,31.321604,Other,Other,Correct,Open or bare land observed.


Point 49 saved successfully!


In [145]:
point_50_lon = validation_df.iloc[49]["Longitude"]
point_50_lat = validation_df.iloc[49]["Latitude"]

point_50 = ee.Geometry.Point([point_50_lon, point_50_lat])

point_50_map = geemap.Map(
    center=[point_50_lat, point_50_lon],
    zoom=18
)

point_50_map.add_basemap("Esri.WorldImagery")
point_50_map.addLayer(
    point_50,
    {"color": "red"},
    "Validation Point 50"
)

point_50_map.addLayerControl()
display(point_50_map)

Map(center=[np.float64(31.364004408570004), np.float64(74.22801650561709)], controls=(WidgetControl(options=['…

In [146]:
validation_df.loc[49, "Actual Class"] = "Vegetation"

predicted = validation_df.loc[49, "Predicted Class"]

validation_df.loc[49, "Correct/Incorrect"] = (
    "Correct" if predicted == "Vegetation" else "Incorrect"
)

validation_df.loc[49, "Notes"] = (
    "Green vegetation or agricultural cover observed."
)

validation_df.to_csv(validation_path, index=False)

display(validation_df.iloc[[49]])

print("Point 50 saved successfully!")

,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
49,74.228017,31.364004,Vegetation,Vegetation,Correct,Green vegetation or agricultural cover observed.


Point 50 saved successfully!


In [147]:
import pandas as pd

validation_df = pd.read_csv(validation_path)

print("Total validation points:", len(validation_df))

print("\nActual Class Distribution:")
print(validation_df["Actual Class"].value_counts(dropna=False))

print("\nValidation Results:")
print(validation_df["Correct/Incorrect"].value_counts(dropna=False))

print("\nMissing Labels:")
print(validation_df["Actual Class"].isna().sum())

display(validation_df.head(50))

Total validation points: 50

Actual Class Distribution:
Actual Class
Vegetation    21
Other         19
Built-up      10
Name: count, dtype: int64

Validation Results:
Correct/Incorrect
Correct         38
Incorrect       11
Needs Review     1
Name: count, dtype: int64

Missing Labels:
0


,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...
3,74.207265,31.334091,Vegetation,Vegetation,Correct,Vegetation and agricultural fields observed in...
4,74.225771,31.345230,Other,Other,Correct,Brown bare soil/open land observed in satellit...
5,74.246162,31.306602,Other,Other,Correct,Plots and brown bare soil observed in satellit...
6,74.251912,31.354752,Other,Other,Correct,Road surface observed; no clear surrounding bu...
7,74.244905,31.348553,Vegetation,Vegetation,Correct,Green vegetation observed around the validatio...
8,74.274729,31.352326,Built-up,Other,Incorrect,House/building observed at the validation point.
9,74.248947,31.349811,Other,Vegetation,Incorrect,Road surface observed at the validation point.


In [148]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Exclude points that need review
valid_df = validation_df[
    validation_df["Correct/Incorrect"].isin(["Correct", "Incorrect"])
].copy()

y_true = valid_df["Actual Class"]
y_pred = valid_df["Predicted Class"]

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

print("Total Points:", len(validation_df))
print("Usable Points:", len(valid_df))
print("Accuracy:", round(accuracy * 100, 2), "%")

# Confusion Matrix
classes = ["Vegetation", "Built-up", "Other"]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=classes
)

print("\nConfusion Matrix:")
display(pd.DataFrame(
    cm,
    index=classes,
    columns=classes
))

# Classification Report
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=classes,
    zero_division=0
))

Total Points: 50
Usable Points: 49
Accuracy: 77.55 %

Confusion Matrix:


,Vegetation,Built-up,Other
Vegetation,19,1,0
Built-up,1,3,6
Other,2,1,16



Classification Report:
              precision    recall  f1-score   support

  Vegetation       0.86      0.95      0.90        20
    Built-up       0.60      0.30      0.40        10
       Other       0.73      0.84      0.78        19

    accuracy                           0.78        49
   macro avg       0.73      0.70      0.70        49
weighted avg       0.76      0.78      0.75        49



In [149]:
from pathlib import Path

training_folder = Path("../data/training_samples")

print("Training folder exists:", training_folder.exists())

if training_folder.exists():
    print("\nFiles available:")
    for file in training_folder.iterdir():
        print(file.name)

Training folder exists: True

Files available:


In [150]:
# Load ESA WorldCover 2021
worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
)

# Convert WorldCover classes into our 3 project classes
# 0 = Vegetation
# 1 = Built-up
# 2 = Other

reference_labels = worldcover.remap(
    [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100],
    [0,  0,  0,  0,  1,  2,  2,  2,  2,  2,   2]
).rename("Land_Class").clip(aoi)

# Prepare Sentinel-2 features
training_image = (
    image_2025
    .select(["B2", "B3", "B4", "B8", "B11", "B12"])
    .addBands(
        image_2025.normalizedDifference(
            ["B8", "B4"]
        ).rename("NDVI")
    )
    .addBands(
        image_2025.normalizedDifference(
            ["B11", "B8"]
        ).rename("NDBI")
    )
    .addBands(reference_labels)
)

# Generate separate training samples
training_samples = training_image.stratifiedSample(
    numPoints=200,
    classBand="Land_Class",
    region=aoi,
    scale=10,
    classValues=[0, 1, 2],
    classPoints=[200, 200, 200],
    seed=42,
    geometries=False,
    tileScale=4
)

# Convert samples into DataFrame
training_features = training_samples.getInfo()["features"]

training_df = pd.DataFrame(
    [feature["properties"] for feature in training_features]
)

# Save training data
training_folder.mkdir(parents=True, exist_ok=True)

training_path = training_folder / "training_samples.csv"

training_df.to_csv(training_path, index=False)

print("Training samples generated:", len(training_df))
print("\nClass distribution:")
print(training_df["Land_Class"].value_counts().sort_index())

print("\nTraining samples saved successfully!")
display(training_df.head())

Training samples generated: 600

Class distribution:
Land_Class
0    200
1    200
2    200
Name: count, dtype: int64

Training samples saved successfully!


,B11,B12,B2,B3,B4,B8,Land_Class,NDBI,NDVI
0,2452.0,1815.0,581.5,962.0,1024.0,2576.5,0,-0.024759,0.431190
1,3482.0,2503.5,844.0,1222.0,1419.0,3229.5,0,0.037622,0.389480
2,2687.5,2020.5,794.5,1172.0,1152.0,2947.5,0,-0.046140,0.437980
3,3163.5,2508.0,984.5,1357.0,1629.0,2954.0,0,0.034246,0.289112
4,2256.0,1433.5,431.5,667.0,714.0,2568.5,0,-0.064774,0.564966


In [151]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Input features
feature_columns = [
    "B2", "B3", "B4", "B8",
    "B11", "B12", "NDVI", "NDBI"
]

X = training_df[feature_columns]
y = training_df["Land_Class"]

# Split training samples: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

# Train model
rf_model.fit(X_train, y_train)

# Evaluate on held-out reference samples
y_pred = rf_model.predict(X_test)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print(
    "\nReference sample accuracy:",
    round(accuracy_score(y_test, y_pred) * 100, 2),
    "%"
)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Vegetation", "Built-up", "Other"]
))

# Save trained model
model_path = Path("../outputs/random_forest_model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(rf_model, model_path)

print("\nRandom Forest model saved successfully!")

Training samples: 480
Testing samples: 120

Reference sample accuracy: 57.5 %

Classification Report:
              precision    recall  f1-score   support

  Vegetation       0.63      0.55      0.59        40
    Built-up       0.56      0.62      0.59        40
       Other       0.55      0.55      0.55        40

    accuracy                           0.57       120
   macro avg       0.58      0.58      0.57       120
weighted avg       0.58      0.57      0.57       120


Random Forest model saved successfully!


In [2]:
import pandas as pd
from pathlib import Path

# Load saved training samples
training_path = Path("../data/training_samples/training_samples.csv")

training_df = pd.read_csv(training_path)

print("Training samples loaded:", len(training_df))
print("Columns:", training_df.columns.tolist())

display(training_df.head())

Training samples loaded: 600
Columns: ['B11', 'B12', 'B2', 'B3', 'B4', 'B8', 'Land_Class', 'NDBI', 'NDVI']


,B11,B12,B2,B3,B4,B8,Land_Class,NDBI,NDVI
0,2452.0,1815.0,581.5,962.0,1024.0,2576.5,0,-0.024759,0.431190
1,3482.0,2503.5,844.0,1222.0,1419.0,3229.5,0,0.037622,0.389480
2,2687.5,2020.5,794.5,1172.0,1152.0,2947.5,0,-0.046140,0.437980
3,3163.5,2508.0,984.5,1357.0,1629.0,2954.0,0,0.034246,0.289112
4,2256.0,1433.5,431.5,667.0,714.0,2568.5,0,-0.064774,0.564966


In [3]:
# Import required libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Define input features
feature_columns = [
    "B2", "B3", "B4", "B8",
    "B11", "B12", "NDVI", "NDBI"
]

# Prepare features and target
X = training_df[feature_columns]
y = training_df["Land_Class"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

# Train the model
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Evaluate the model
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

print(
    "\nReference Sample Accuracy:",
    round(accuracy_score(y_test, y_pred) * 100, 2),
    "%"
)

# Classification report
print("\nClassification Report:")

print(classification_report(
    y_test,
    y_pred,
    target_names=["Vegetation", "Built-up", "Other"]
))

# Save trained model
model_path = Path("../outputs/random_forest_model.pkl")

model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(rf_model, model_path)

print("\nRandom Forest Model Saved Successfully!")

Training Samples: 480
Testing Samples: 120

Reference Sample Accuracy: 57.5 %

Classification Report:
              precision    recall  f1-score   support

  Vegetation       0.63      0.55      0.59        40
    Built-up       0.56      0.62      0.59        40
       Other       0.55      0.55      0.55        40

    accuracy                           0.57       120
   macro avg       0.58      0.58      0.57       120
weighted avg       0.58      0.57      0.57       120


Random Forest Model Saved Successfully!


In [6]:
print("Earth Engine image loaded:", "image_2025" in globals())
print("Validation data loaded:", "validation_df" in globals())
print("Random Forest model loaded:", "rf_model" in globals())
print("Training data loaded:", "training_df" in globals())

Earth Engine image loaded: False
Validation data loaded: False
Random Forest model loaded: True
Training data loaded: True


In [7]:
import ee
import pandas as pd
from pathlib import Path

# Initialize Google Earth Engine
PROJECT_ID = "punjab-geoguardian-515011"

ee.Initialize(project=PROJECT_ID)

# Restore Area of Interest
aoi = ee.Geometry.Rectangle([
    74.20, 31.30, 74.28, 31.38
])

# Cloud masking function
def mask_s2_clouds(image):
    scl = image.select("SCL")

    cloud_mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    return image.updateMask(cloud_mask)

# Restore 2025 Sentinel-2 imagery
sentinel_2025 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate("2025-10-01", "2026-01-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

image_2025 = (
    sentinel_2025
    .map(mask_s2_clouds)
    .median()
    .clip(aoi)
)

# Restore manual validation data
validation_path = Path(
    "../data/validation_data/validation_points.csv"
)

validation_df = pd.read_csv(validation_path)

# Check restored variables
print("Earth Engine initialized successfully!")
print("2025 images:", sentinel_2025.size().getInfo())
print("Validation points:", len(validation_df))
print("Random Forest model:", "rf_model" in globals())

display(validation_df.head())

Earth Engine initialized successfully!
2025 images: 32
Validation points: 50
Random Forest model: True


,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes
0,74.202055,31.322772,Vegetation,Vegetation,Correct,Agricultural fields and vegetation observed in...
1,74.248588,31.349092,Built-up,Other,Incorrect,Buildings observed using high-resolution satel...
2,74.225950,31.323490,Vegetation,Vegetation,Correct,Vegetation and green agricultural fields obser...
3,74.207265,31.334091,Vegetation,Vegetation,Correct,Vegetation and agricultural fields observed in...
4,74.225771,31.345230,Other,Other,Correct,Brown bare soil/open land observed in satellit...


In [8]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Prepare Sentinel-2 features
feature_image = (
    image_2025
    .select(["B2", "B3", "B4", "B8", "B11", "B12"])
    .addBands(
        image_2025.normalizedDifference(
            ["B8", "B4"]
        ).rename("NDVI")
    )
    .addBands(
        image_2025.normalizedDifference(
            ["B11", "B8"]
        ).rename("NDBI")
    )
)

# Create Earth Engine validation points
validation_features = []

for index, row in validation_df.iterrows():

    point = ee.Geometry.Point([
        row["Longitude"],
        row["Latitude"]
    ])

    feature = ee.Feature(
        point,
        {"row_id": int(index)}
    )

    validation_features.append(feature)

validation_fc = ee.FeatureCollection(validation_features)

# Extract satellite features at validation points
sampled_validation = feature_image.sampleRegions(
    collection=validation_fc,
    properties=["row_id"],
    scale=10,
    geometries=False,
    tileScale=4
)

# Convert extracted features into DataFrame
sampled_data = sampled_validation.getInfo()["features"]

sampled_df = pd.DataFrame([
    feature["properties"]
    for feature in sampled_data
])

print("Successfully sampled points:", len(sampled_df))

# Generate Random Forest predictions
feature_columns = [
    "B2", "B3", "B4", "B8",
    "B11", "B12", "NDVI", "NDBI"
]

sampled_df["RF Prediction"] = rf_model.predict(
    sampled_df[feature_columns]
)

# Convert numeric predictions into class names
class_names = {
    0: "Vegetation",
    1: "Built-up",
    2: "Other"
}

sampled_df["RF Predicted Class"] = (
    sampled_df["RF Prediction"].map(class_names)
)

# Add predictions to original validation data
validation_df["RF Predicted Class"] = pd.NA

for _, row in sampled_df.iterrows():
    validation_df.loc[
        int(row["row_id"]),
        "RF Predicted Class"
    ] = row["RF Predicted Class"]

# Evaluate only manually labeled points
valid_df = validation_df[
    validation_df["Correct/Incorrect"].isin(
        ["Correct", "Incorrect"]
    )
    & validation_df["RF Predicted Class"].notna()
].copy()

y_true = valid_df["Actual Class"]
y_pred = valid_df["RF Predicted Class"]

classes = ["Vegetation", "Built-up", "Other"]

print("\nIndependent Validation Results")
print("Usable points:", len(valid_df))

print(
    "Random Forest Accuracy:",
    round(accuracy_score(y_true, y_pred) * 100, 2),
    "%"
)

print("\nConfusion Matrix:")

display(pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=classes),
    index=classes,
    columns=classes
))

print("\nClassification Report:")

print(classification_report(
    y_true,
    y_pred,
    labels=classes,
    zero_division=0
))

# Save updated validation results
validation_df.to_csv(validation_path, index=False)

print("\nValidation results saved successfully!")

display(
    validation_df[
        ["Actual Class", "Predicted Class", "RF Predicted Class"]
    ].head(10)
)

Successfully sampled points: 50

Independent Validation Results
Usable points: 49
Random Forest Accuracy: 61.22 %

Confusion Matrix:


,Vegetation,Built-up,Other
Vegetation,18,2,0
Built-up,1,4,5
Other,5,6,8



Classification Report:
              precision    recall  f1-score   support

  Vegetation       0.75      0.90      0.82        20
    Built-up       0.33      0.40      0.36        10
       Other       0.62      0.42      0.50        19

    accuracy                           0.61        49
   macro avg       0.57      0.57      0.56        49
weighted avg       0.61      0.61      0.60        49


Validation results saved successfully!


,Actual Class,Predicted Class,RF Predicted Class
0,Vegetation,Vegetation,Vegetation
1,Built-up,Other,Other
2,Vegetation,Vegetation,Vegetation
3,Vegetation,Vegetation,Vegetation
4,Other,Other,Vegetation
5,Other,Other,Other
6,Other,Other,Other
7,Vegetation,Vegetation,Vegetation
8,Built-up,Other,Other
9,Other,Vegetation,Built-up


In [10]:
# Identify Random Forest misclassified points
error_df = validation_df[
    validation_df["Correct/Incorrect"].isin(
        ["Correct", "Incorrect"]
    )
    & validation_df["RF Predicted Class"].notna()
    & (
        validation_df["Actual Class"]
        != validation_df["RF Predicted Class"]
    )
].copy()

# Select columns for error analysis
error_analysis = error_df[
    [
        "Longitude",
        "Latitude",
        "Actual Class",
        "RF Predicted Class",
        "Notes"
    ]
].copy()

# Add error type
error_analysis["Error Type"] = (
    error_analysis["Actual Class"]
    + " classified as "
    + error_analysis["RF Predicted Class"]
)

# Save error analysis
error_path = Path(
    "../data/validation_data/error_analysis.csv"
)

error_path.parent.mkdir(parents=True, exist_ok=True)

error_analysis.to_csv(error_path, index=False)

print("Total error cases:", len(error_analysis))
print("Error analysis saved successfully!")

display(error_analysis)

Total error cases: 19
Error analysis saved successfully!


,Longitude,Latitude,Actual Class,RF Predicted Class,Notes,Error Type
1,74.248588,31.349092,Built-up,Other,Buildings observed using high-resolution satel...,Built-up classified as Other
4,74.225771,31.345230,Other,Vegetation,Brown bare soil/open land observed in satellit...,Other classified as Vegetation
8,74.274729,31.352326,Built-up,Other,House/building observed at the validation point.,Built-up classified as Other
9,74.248947,31.349811,Other,Built-up,Road surface observed at the validation point.,Other classified as Built-up
14,74.215260,31.331306,Other,Vegetation,Rural open land with an unpaved dirt track obs...,Other classified as Vegetation
15,74.230442,31.346667,Vegetation,Built-up,Green grassland/agricultural field boundary ad...,Vegetation classified as Built-up
16,74.275627,31.321694,Vegetation,Built-up,Agricultural field boundary (field bund) obser...,Vegetation classified as Built-up
19,74.233856,31.319897,Built-up,Other,Paved sidewalk or road-side urban infrastructu...,Built-up classified as Other
20,74.255146,31.311004,Other,Built-up,Location does not clearly show vegetation or b...,Other classified as Built-up
26,74.259637,31.338941,Other,Vegetation,Area does not clearly show vegetation or built...,Other classified as Vegetation


In [11]:
# Prepare features for Earth Engine classification

feature_columns = [
    "B2", "B3", "B4", "B8",
    "B11", "B12", "NDVI", "NDBI"
]

# Convert training samples into Earth Engine features
training_records = training_df.to_dict(orient="records")

ee_training_features = []

for record in training_records:
    properties = {
        column: float(record[column])
        for column in feature_columns
    }

    properties["Land_Class"] = int(record["Land_Class"])

    ee_training_features.append(
        ee.Feature(None, properties)
    )

ee_training_fc = ee.FeatureCollection(
    ee_training_features
)

# Train Earth Engine Random Forest
rf_classifier_ee = ee.Classifier.smileRandomForest(
    numberOfTrees=300,
    seed=42
).train(
    features=ee_training_fc,
    classProperty="Land_Class",
    inputProperties=feature_columns
)

print("Earth Engine Random Forest trained successfully!")

# Prepare Sentinel-2 image collections
def get_sentinel_composite(start_date, end_date):

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)
        )
    )

    return (
        collection
        .map(mask_s2_clouds)
        .median()
        .clip(aoi)
    )

# Restore both years
image_2019 = get_sentinel_composite(
    "2019-10-01", "2020-01-01"
)

image_2025 = get_sentinel_composite(
    "2025-10-01", "2026-01-01"
)

# Function to prepare image features
def prepare_features(image):

    return (
        image.select([
            "B2", "B3", "B4",
            "B8", "B11", "B12"
        ])
        .addBands(
            image.normalizedDifference(
                ["B8", "B4"]
            ).rename("NDVI")
        )
        .addBands(
            image.normalizedDifference(
                ["B11", "B8"]
            ).rename("NDBI")
        )
    )

# Prepare both years
features_2019 = prepare_features(image_2019)
features_2025 = prepare_features(image_2025)

# Classify both years
classified_rf_2019 = (
    features_2019
    .classify(rf_classifier_ee)
    .rename("Land_Class")
    .clip(aoi)
)

classified_rf_2025 = (
    features_2025
    .classify(rf_classifier_ee)
    .rename("Land_Class")
    .clip(aoi)
)

print("2019 Random Forest classification completed!")
print("2025 Random Forest classification completed!")

Earth Engine Random Forest trained successfully!
2019 Random Forest classification completed!
2025 Random Forest classification completed!


In [13]:
import geemap
import ee

print("geemap imported successfully!")

geemap imported successfully!


In [14]:
# Create map
Map = geemap.Map(center=[31.34, 74.24], zoom=12)

# Define class colors
class_palette = [
    "228B22",  # Vegetation - Green
    "FF0000",  # Built-up - Red
    "D2B48C"   # Other - Brown
]

# Display 2019 classification
Map.addLayer(
    classified_rf_2019,
    {
        "min": 0,
        "max": 2,
        "palette": class_palette
    },
    "Random Forest Classification 2019"
)

# Display 2025 classification
Map.addLayer(
    classified_rf_2025,
    {
        "min": 0,
        "max": 2,
        "palette": class_palette
    },
    "Random Forest Classification 2025"
)

# Display AOI boundary
Map.addLayer(
    ee.FeatureCollection([ee.Feature(aoi)]).style(
        color="FFFFFF",
        fillColor="00000000",
        width=2
    ),
    {},
    "AOI Boundary"
)

Map

Map(center=[31.34, 74.24], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [15]:
# Show only 2019 classification
Map.layers[-1].visible = False
Map.layers[-2].visible = False

print("2019 map selected")

2019 map selected


In [16]:
for layer in Map.layers:
    print(layer.name)

OpenStreetMap.HOT
Random Forest Classification 2019
Random Forest Classification 2025
AOI Boundary


In [17]:
# Show only 2019 classification
for layer in Map.layers:
    if layer.name == "Random Forest Classification 2019":
        layer.visible = True
    elif layer.name == "Random Forest Classification 2025":
        layer.visible = False

Map

Map(bottom=428330.0, center=[31.354223315787646, 74.24028789748105], controls=(WidgetControl(options=['positio…

In [19]:
# Show only 2025 classification
for layer in Map.layers:
    if layer.name == "Random Forest Classification 2019":
        layer.visible = False
    elif layer.name == "Random Forest Classification 2025":
        layer.visible = True

Map

Map(bottom=108296.0, center=[30.18787014479982, 74.5305633544922], controls=(WidgetControl(options=['position'…

In [20]:
# 1. Calculate possible vegetation-to-built-up change

vegetation_2019 = classified_rf_2019.eq(0)
builtup_2025 = classified_rf_2025.eq(1)

rf_change_map = (
    vegetation_2019
    .And(builtup_2025)
    .selfMask()
    .rename("Vegetation_to_Builtup")
)

# 2. Calculate changed area in square kilometres

area_image = ee.Image.pixelArea().updateMask(rf_change_map)

change_area = area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=10,
    maxPixels=1e13,
    tileScale=4
)

change_area_km2 = (
    change_area.get("area").getInfo() / 1_000_000
)

print("Random Forest Change Detection Results")
print("---------------------------------------")
print(
    "Possible Vegetation-to-Built-up Change:",
    round(change_area_km2, 4),
    "km²"
)

# 3. Add change map to existing map

Map.addLayer(
    rf_change_map,
    {"palette": ["FFFF00"]},
    "Possible Vegetation to Built-up Change"
)

Map

Random Forest Change Detection Results
---------------------------------------
Possible Vegetation-to-Built-up Change: 5.5107 km²


Map(bottom=108296.0, center=[30.18787014479982, 74.5305633544922], controls=(WidgetControl(options=['position'…

In [21]:
# Calculate area of each land-cover class

def calculate_class_areas(classified_image, year):

    area_image = ee.Image.pixelArea().divide(1_000_000)

    results = []

    classes = [
        (0, "Vegetation"),
        (1, "Built-up"),
        (2, "Other")
    ]

    for class_id, class_name in classes:

        class_area = (
            area_image
            .updateMask(classified_image.eq(class_id))
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=aoi,
                scale=10,
                maxPixels=1e13,
                tileScale=4
            )
            .get("area")
            .getInfo()
        )

        results.append({
            "Year": year,
            "Class": class_name,
            "Area_km2": round(class_area or 0, 4)
        })

    return results


# Calculate areas for both years

areas_2019 = calculate_class_areas(
    classified_rf_2019, 2019
)

areas_2025 = calculate_class_areas(
    classified_rf_2025, 2025
)

# Create statistics table

rf_area_df = pd.DataFrame(
    areas_2019 + areas_2025
)

display(rf_area_df)

# Save CSV file

stats_path = Path(
    "../outputs/random_forest_area_statistics.csv"
)

stats_path.parent.mkdir(parents=True, exist_ok=True)

rf_area_df.to_csv(stats_path, index=False)

print("Area statistics saved successfully!")

,Year,Class,Area_km2
0,2019,Vegetation,30.4550
1,2019,Built-up,16.1040
2,2019,Other,20.9648
3,2025,Vegetation,30.7272
4,2025,Built-up,19.4656
5,2025,Other,17.3309


Area statistics saved successfully!


In [23]:
from pathlib import Path
import geemap

output_folder = Path("../outputs/rgb_maps")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "RGB_2019.tif"

geemap.ee_export_image(
    rgb_2019.visualize(
        bands=["B4", "B3", "B2"],
        min=0,
        max=3000
    ),
    filename=str(output_file),
    scale=20,       # 10m ki jagah 20m, smaller download
    region=aoi,
    file_per_band=False,
    timeout=300
)

print("RGB 2019 export completed!")


Generating URL ...
Please wait ...
Data downloaded to d:\Punjab-GeoGuardian\outputs\rgb_maps\RGB_2019.tif
RGB 2019 export completed!


In [24]:
output_file = output_folder / "RGB_2025.tif"

geemap.ee_export_image(
    rgb_2025.visualize(
        bands=["B4", "B3", "B2"],
        min=0,
        max=3000
    ),
    filename=str(output_file),
    scale=20,
    region=aoi,
    file_per_band=False,
    timeout=300
)

print("RGB 2025 export completed!")

Generating URL ...
Please wait ...
Data downloaded to d:\Punjab-GeoGuardian\outputs\rgb_maps\RGB_2025.tif
RGB 2025 export completed!


In [ ]:
output_folder = Path("../outputs/ndvi_maps")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "NDVI_2019.tif"

geemap.ee_export_image(
    ndvi_2019.visualize(
        min=-0.2,
        max=0.8,
        palette=["blue", "white", "green"]
    ),
    filename=str(output_file),
    scale=20,
    region=aoi,
    file_per_band=False,
    timeout=300
)

print("NDVI 2019 export completed!")

Generating URL ...
Please wait ...
Data downloaded to d:\Punjab-GeoGuardian\outputs\ndvi_maps\NDVI_2019.tif
NDVI 2019 export completed!


In [26]:
output_file = output_folder / "NDVI_2025.tif"

geemap.ee_export_image(
    ndvi_2025.visualize(
        min=-0.2,
        max=0.8,
        palette=["blue", "white", "green"]
    ),
    filename=str(output_file),
    scale=20,
    region=aoi,
    file_per_band=False,
    timeout=300
)

print("NDVI 2025 export completed!")

Generating URL ...
Please wait ...
Data downloaded to d:\Punjab-GeoGuardian\outputs\ndvi_maps\NDVI_2025.tif
NDVI 2025 export completed!


In [27]:
output_folder = Path("../outputs/ndbi_maps")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "NDBI_2019.tif"

geemap.ee_export_image(
    ndbi_2019.visualize(
        min=-0.5,
        max=0.5,
        palette=["blue", "white", "red"]
    ),
    filename=str(output_file),
    scale=20,
    region=aoi,
    file_per_band=False,
    timeout=300
)

print("NDBI 2019 export completed!")

Generating URL ...
Please wait ...
Data downloaded to d:\Punjab-GeoGuardian\outputs\ndbi_maps\NDBI_2019.tif
NDBI 2019 export completed!


In [28]:
output_file = output_folder / "NDBI_2025.tif"

geemap.ee_export_image(
    ndbi_2025.visualize(
        min=-0.5,
        max=0.5,
        palette=["blue", "white", "red"]
    ),
    filename=str(output_file),
    scale=20,
    region=aoi,
    file_per_band=False,
    timeout=300
)

print("NDBI 2025 export completed!")

Generating URL ...
Please wait ...
Data downloaded to d:\Punjab-GeoGuardian\outputs\ndbi_maps\NDBI_2025.tif
NDBI 2025 export completed!


In [29]:
output_folder = Path("../outputs/classification_maps")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "RF_Classification_2019.tif"

geemap.ee_export_image(
    classified_rf_2019.visualize(
        min=0,
        max=2,
        palette=["228B22", "FF0000", "D2B48C"]
    ),
    filename=str(output_file),
    scale=20,
    region=aoi,
    file_per_band=False,
    timeout=300
)

print("RF Classification 2019 export completed!")

Generating URL ...
Please wait ...
Data downloaded to d:\Punjab-GeoGuardian\outputs\classification_maps\RF_Classification_2019.tif
RF Classification 2019 export completed!


In [30]:
output_file = output_folder / "RF_Classification_2025.tif"

geemap.ee_export_image(
    classified_rf_2025.visualize(
        min=0,
        max=2,
        palette=["228B22", "FF0000", "D2B48C"]
    ),
    filename=str(output_file),
    scale=20,
    region=aoi,
    file_per_band=False,
    timeout=300
)

print("RF Classification 2025 export completed!")

Generating URL ...
Please wait ...
Data downloaded to d:\Punjab-GeoGuardian\outputs\classification_maps\RF_Classification_2025.tif
RF Classification 2025 export completed!


In [31]:
output_folder = Path("../outputs/change_detection")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "RF_Vegetation_to_Builtup_Change.tif"

geemap.ee_export_image(
    rf_change_map.visualize(
        palette=["FFFF00"]
    ),
    filename=str(output_file),
    scale=20,
    region=aoi,
    file_per_band=False,
    timeout=300
)

print("Change Detection map export completed!")

Generating URL ...
Please wait ...
Data downloaded to d:\Punjab-GeoGuardian\outputs\change_detection\RF_Vegetation_to_Builtup_Change.tif
Change Detection map export completed!


In [32]:
from pathlib import Path

output_dir = Path("../outputs")

for file in sorted(output_dir.rglob("*.tif")):
    print(file)

print("\nTotal TIFF maps:", len(list(output_dir.rglob("*.tif"))))

..\outputs\change_detection\RF_Vegetation_to_Builtup_Change.tif
..\outputs\classification_maps\RF_Classification_2019.tif
..\outputs\classification_maps\RF_Classification_2025.tif
..\outputs\ndbi_maps\NDBI_2019.tif
..\outputs\ndbi_maps\NDBI_2025.tif
..\outputs\ndvi_maps\NDVI_2019.tif
..\outputs\ndvi_maps\NDVI_2025.tif
..\outputs\rgb_maps\RGB_2019.tif
..\outputs\rgb_maps\RGB_2025.tif

Total TIFF maps: 9


In [33]:
import importlib.util

if importlib.util.find_spec("streamlit"):
    print("Streamlit is already installed!")
else:
    print("Streamlit is not installed.")

Streamlit is already installed!


In [34]:
print("Validation rows:", len(validation_df))
print("Validation columns:", validation_df.columns.tolist())
display(validation_df.tail())

Validation rows: 50
Validation columns: ['Longitude', 'Latitude', 'Actual Class', 'Predicted Class', 'Correct/Incorrect', 'Notes', 'RF Predicted Class']


,Longitude,Latitude,Actual Class,Predicted Class,Correct/Incorrect,Notes,RF Predicted Class
45,74.219483,31.329240,Vegetation,Vegetation,Correct,Green vegetation or agricultural cover observed.,Vegetation
46,74.265836,31.333192,Built-up,Built-up,Correct,Building or urban infrastructure observed.,Other
47,74.243108,31.346218,Other,Other,Correct,Open or bare land observed.,Other
48,74.237359,31.321604,Other,Other,Correct,Open or bare land observed.,Built-up
49,74.228017,31.364004,Vegetation,Vegetation,Correct,Green vegetation or agricultural cover observed.,Vegetation


In [35]:
saved_df = pd.read_csv(validation_path)

print("Saved CSV rows:", len(saved_df))

Saved CSV rows: 50


In [36]:
print("Rule-based validation:")
print(validation_df["Correct/Incorrect"].value_counts())

accuracy = (
    (validation_df["Correct/Incorrect"] == "Correct").sum()
    / len(validation_df)
) * 100

print(f"Updated Rule-based Accuracy: {accuracy:.2f}%")

Rule-based validation:
Correct/Incorrect
Correct         38
Incorrect       11
Needs Review     1
Name: count, dtype: int64
Updated Rule-based Accuracy: 76.00%


In [37]:
rf_correct = (
    validation_df["Actual Class"] == validation_df["RF Predicted Class"]
).sum()

rf_accuracy = (rf_correct / len(validation_df)) * 100

print("RF Correct:", rf_correct)
print("RF Incorrect:", len(validation_df) - rf_correct)
print(f"Updated RF Accuracy: {rf_accuracy:.2f}%")

RF Correct: 30
RF Incorrect: 20
Updated RF Accuracy: 60.00%


In [39]:
from pathlib import Path

matches = list(Path("..").rglob("error_analysis.csv"))

for file in matches:
    print(file.resolve())

D:\Punjab-GeoGuardian\outputs\error_analysis.csv
D:\Punjab-GeoGuardian\data\validation_data\error_analysis.csv


In [40]:
from pathlib import Path
import pandas as pd

files = [
    Path(r"D:\Punjab-GeoGuardian\outputs\error_analysis.csv"),
    Path(r"D:\Punjab-GeoGuardian\data\validation_data\error_analysis.csv")
]

for file in files:
    df = pd.read_csv(file)
    print(f"\n{file}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())


D:\Punjab-GeoGuardian\outputs\error_analysis.csv
Rows: 4
Columns: ['Point', 'Actual Class', 'Predicted Class', 'Longitude', 'Latitude', 'Error Type', 'Possible Reason']

D:\Punjab-GeoGuardian\data\validation_data\error_analysis.csv
Rows: 19
Columns: ['Longitude', 'Latitude', 'Actual Class', 'RF Predicted Class', 'Notes', 'Error Type']


In [41]:
error_df = pd.read_csv(
    r"D:\Punjab-GeoGuardian\data\validation_data\error_analysis.csv"
)

print("Error cases:", len(error_df))
display(error_df.head())

Error cases: 19


,Longitude,Latitude,Actual Class,RF Predicted Class,Notes,Error Type
0,74.248588,31.349092,Built-up,Other,Buildings observed using high-resolution satel...,Built-up classified as Other
1,74.225771,31.345230,Other,Vegetation,Brown bare soil/open land observed in satellit...,Other classified as Vegetation
2,74.274729,31.352326,Built-up,Other,House/building observed at the validation point.,Built-up classified as Other
3,74.248947,31.349811,Other,Built-up,Road surface observed at the validation point.,Other classified as Built-up
4,74.215260,31.331306,Other,Vegetation,Rural open land with an unpaved dirt track obs...,Other classified as Vegetation


In [42]:
print("RF incorrect points:", 
      (validation_df["Actual Class"] != validation_df["RF Predicted Class"]).sum())

print("Error analysis records:", len(error_df))

RF incorrect points: 20
Error analysis records: 19


In [44]:
print("Missing error records:", len(missing_error))
print("Available columns:", missing_error.columns.tolist())
display(missing_error)

Missing error records: 1
Available columns: ['Longitude', 'Latitude', 'Actual Class_x', 'Predicted Class', 'Correct/Incorrect', 'Notes_x', 'RF Predicted Class_x', 'Actual Class_y', 'RF Predicted Class_y', 'Notes_y', 'Error Type', '_merge']


,Longitude,Latitude,Actual Class_x,Predicted Class,Correct/Incorrect,Notes_x,RF Predicted Class_x,Actual Class_y,RF Predicted Class_y,Notes_y,Error Type,_merge
4,74.254427,31.345769,Vegetation,Other,Needs Review,"Possible vegetation observed, but imagery is b...",Built-up,NaN,NaN,NaN,NaN,left_only


In [45]:
summary = pd.DataFrame({
    "Model": ["Rule-based", "Random Forest"],
    "Correct": [38, 30],
    "Incorrect": [12, 20],
    "Accuracy (%)": [76.00, 60.00]
})

display(summary)

summary.to_csv(
    r"D:\Punjab-GeoGuardian\reports\updated_validation_summary.csv",
    index=False
)

print("Updated validation summary saved!")

,Model,Correct,Incorrect,Accuracy (%)
0,Rule-based,38,12,76.0
1,Random Forest,30,20,60.0


Updated validation summary saved!
